# Full Title: Type 2 Diabetes Vascular Niche Characterization — Cell Typing, Spatial Distance Mapping, and ECM/BM Module Analysis from MERSCOPE Spatial Transcriptomics## T2D Vascular Niche Characterization: Cell Typing, Spatial Distance Mapping, and ECM/BM Module Analysis

**Input:** Per-sample islet and exocrine AnnData objects (HuP51A, HuP61A, HuP74A, HuP93A, HuP97A)

**Outputs:**
- `t2dcells_allsamples.h5ad` → `adata` (full T2D dataset, all cell types)
- `t2dcells_allsamples_filteredlabeled.h5ad` → `adata` (annotated)
- `t2dvascells_allsamples_filteredlabeled.h5ad` → `vasc_t2df` (final vascular object)

---

### Pipeline Overview

1. Load per-sample islet + exocrine objects, attach metadata
2. Subsample exocrine cells (fraction=0.028), concatenate → `adata`
3. Normalize, log1p transform
4. Concord batch integration (domain_key = Sample) + UMAP
5. Leiden clustering (res=1.0) on Concord neighbor graph
6. Cluster QC: dotplots, marker UMAPs, Wilcoxon DEGs per cluster
7. Clusters merged into broad cell types → `merged_cell_type`, `specific_cell_type`
8. Islet ROI geometry parsed from WKT polygons → area, perimeter, circularity
9. Cell boundary polygons loaded from Parquet files per donor
10. Spatial distance calculations: islet boundary, endothelial, mesenchymal, beta cells
11. Mesenchymal + endothelial cells subset → `vasc_t2d`
12. Vascular Leiden sub-clustering (res=1.0) → `vasc_leiden`
13. Marker-guided cluster merging into 4 final vascular cell types
14. Islet fibroblasts relabeled as Islet-associated Fibroblasts by Location
15. Ambiguous clusters (leiden 3, 4) removed → `vasc_t2df`
16. UMAP recomputed on vascular subset (Concord_UMAPmRU)
17. DEG comparisons: Pericytes vs EC, Pericytes vs IAF, Fibroblasts vs IAF
18. BM gene module scoring and matrixplots (islet compartment only)
19. Spatial enrichment: KDE, boxplots, pericyte coverage, donor-level stats
20. Final objects saved

---

### Cluster → Cell Type Mapping (Full Dataset)

| Leiden Clusters | Final Label |
|---|---|
| 3, 5, 16, 9 | Alpha Cells |
| 13, 2, 14 | Beta Cells |
| 0, 4 | Acinar Cells |
| 7, 8 | Endothelial Cells |
| 11, 15 | Immune Cells |
| 1 | Ductal Cells |
| 6 | Mesenchymal Cells |
| 10 | Delta Cells |
| 12 | PPY Cells |

### Vascular Cluster → Cell Type Mapping (`vasc_leiden`)

| Leiden Clusters | Final Label |
|---|---|
| 2, 5, 7, 10, 11 | Endothelial Cells |
| 0, 6, 12 | Fibroblasts |
| 1, 9 | Pericytes |
| 8 | Islet-associated Fibroblasts |
| 3, 4 | Removed (ambiguous) |
| Fibroblasts in Location == Islet | → Islet-associated Fibroblasts |

**Note:** Islet-associated Fibroblasts are defined as fibroblasts
physically located within the islet compartment (Location == "Islet"),
confirmed by spatial distance metrics and DEG analysis
(Wilcoxon, use_raw=False).

---

### Spatial Distance Columns

| Column | Reference Cell Type | Notes |
|---|---|---|
| `dist_edge_islet_um` | Islet ROI polygon boundary | Negative = inside islet |
| `dist_edge_endo_um` | Endothelial Cells | Edge-to-edge, px × 0.108 |
| `dist_edge_mesenchymal_um` | Mesenchymal Cells | Edge-to-edge, px × 0.108 |
| `dist_edge_beta_um` | Beta Cells | Edge-to-edge, px × 0.108 |
| `dist_edge_pericyte_um` | Pericytes | Computed on islet_adata_vasc only |

---

### Key Output Columns

| Column | Object | Description |
|---|---|---|
| `merged_cell_type` | adata | Broad cell type labels (all cells) |
| `specific_cell_type` | adata | Finer cell type labels (all cells) |
| `vasc_leiden` | vasc_t2d | Raw Leiden clusters (res=1.0) |
| `cell_type` | vasc_t2df | Final 4-category vascular labels |
| `cell_type_refined` | adata | Full dataset with vascular labels transferred back |

---

### DEG Comparisons Run

| Comparison | Reference | Object | Notes |
|---|---|---|---|
| Pericytes vs Endothelial Cells | Endothelial Cells | vasc_t2df | Islet only |
| Endothelial Cells vs Pericytes | Pericytes | vasc_t2df | Islet only |
| IAF vs Pericytes | Pericytes | vasc_t2df | Islet only |
| Fibroblasts vs IAF | IAF | vasc_t2df | Islet only |
| IAF vs Fibroblasts | Fibroblasts | vasc_t2df | Islet only |
| Pericytes: Islet vs Exocrine | Exocrine | pericytes_allislet | Location DEG |
| T2D vs ND per cell type | ND | merged_vasc | All + by Location |

---

### BM Gene Module

| Gene Set | Genes |
|---|---|
| Type IV Collagens | COL4A1, COL4A2, COL4A3 |
| Laminins | LAMA4, LAMA5, LAMB1, LAMB2, LAMC1, LAMC3 |

Scored via `sc.tl.score_genes`, compared between:
- Pericytes vs Islet-associated Fibroblasts (MWU + paired Wilcoxon)
- Pericytes vs Endothelial Cells (MWU + paired Wilcoxon)

---

### Donor Metadata

| Sample | Sex | Age | BMI | Health Status |
|---|---|---|---|---|
| HuP51A | F | 74 | 15.1 | T2D |
| HuP61A | M | 69 | 26.8 | T2D |
| HuP74A | F | 58 | 32.9 | T2D |
| HuP93A | M | 71 | 24.7 | T2D |
| HuP97A | F | 69 | 31.2 | T2D |

---

### Color Palette

| Cell Type | Hex |
|---|---|
| Pericytes | #FD15FA |
| Islet-associated Fibroblasts | #f9a942 |
| Endothelial Cells | #08c93b |
| Fibroblasts | #0eecf8 |
| Islet | #3B0A45 |
| Exocrine | #CFA0E9 |

In [ ]:
import concord as ccd
import scanpy as sc
import torch
import os
import pandas as pd
import numpy as np
import anndata as ad

# SECTION 1 — DATA LOADING & METADATA ANNOTATION
 Loads per-sample islet and exocrine AnnData objects from disk,
 assigns spatial compartment (Location: Islet / Exocrine),
 and attaches donor-level metadata: Sample ID, Health_Status,
 Sex, Age, BMI, Ethnicity.

 Exocrine cells are subsampled (fraction=0.028) to match
 islet cell numbers before concatenation.

Key outputs:
   adata  — full T2D dataset (all cell types, both compartments)


In [ ]:
adata = sc.read_h5ad('/Users/kmeneses/Desktop/updated_data/t2dcells_allsamples.h5ad')
adata

In [ ]:
os.chdir('/Volumes/Samsung_4TB/Desktop_copy/Islet_exorine_andatas')
adata51_in = sc.read_h5ad('HuP51A_isletcells.h5ad')
adata61_in = sc.read_h5ad('HuP61A_isletcells.h5ad')
adata74_in = sc.read_h5ad('HuP74A_isletcells.h5ad')
adata93_in = sc.read_h5ad('HuP93A_isletcells.h5ad')
adata97_in = sc.read_h5ad('HuP97A_isletcells.h5ad')




In [ ]:
adata51_in

In [ ]:
os.chdir('/Volumes/Samsung_4TB/Desktop_copy/Islet_exorine_andatas')
adata51_out = sc.read_h5ad('HuP51A_NONisletcells.h5ad')
adata61_out = sc.read_h5ad('HuP61A_NONisletcells.h5ad')
adata74_out = sc.read_h5ad('HuP74A_NONisletcells.h5ad')
adata93_out = sc.read_h5ad('HuP93A_NONisletcells.h5ad')
adata97_out = sc.read_h5ad('HuP97A_NONisletcells.h5ad')




In [ ]:
adata97_out

In [ ]:
adata51_in.obs['Location'] = 'Islet'
adata61_in.obs['Location'] = 'Islet'
adata74_in.obs['Location'] = 'Islet'
adata93_in.obs['Location'] = 'Islet'
adata97_in.obs['Location'] = 'Islet'



adata51_in.obs['Sample'] = 'HuP51A'
adata61_in.obs['Sample'] = 'HuP61A'
adata74_in.obs['Sample'] = 'HuP74A'
adata93_in.obs['Sample'] = 'HuP93A'
adata97_in.obs['Sample'] = 'HuP97A'


adata51_in.obs['Health_Status'] = 'T2D'
adata61_in.obs['Health_Status'] = 'T2D'
adata74_in.obs['Health_Status'] = 'T2D'
adata93_in.obs['Health_Status'] = 'T2D'
adata97_in.obs['Health_Status'] = 'T2D'


adata51_in.obs['Sex'] = 'F'
adata61_in.obs['Sex'] = 'M'
adata74_in.obs['Sex'] = 'F'
adata93_in.obs['Sex'] = 'M'
adata97_in.obs['Sex'] = 'F'


adata51_in.obs['Age'] = '74'
adata61_in.obs['Age'] = '69'
adata74_in.obs['Age'] = '58'
adata93_in.obs['Age'] = '71'
adata97_in.obs['Age'] = '69'


adata51_in.obs['BMI'] = '15.1'
adata61_in.obs['BMI'] = '26.8'
adata74_in.obs['BMI'] = '32.9'
adata93_in.obs['BMI'] = '24.7'
adata97_in.obs['BMI'] = '31.2'


adata51_in.obs['Ethnicity'] = 'Caucasian'
adata61_in.obs['Ethnicity'] = 'Caucasian'
adata74_in.obs['Ethnicity'] = 'Caucasian'
adata93_in.obs['Ethnicity'] = 'Caucasian'
adata97_in.obs['Ethnicity'] = 'Caucasian'


In [ ]:
adata51_out.obs['Location'] = 'Exocrine'
adata61_out.obs['Location'] = 'Exocrine'
adata74_out.obs['Location'] = 'Exocrine'
adata93_out.obs['Location'] = 'Exocrine'
adata97_out.obs['Location'] = 'Exocrine'



adata51_out.obs['Sample'] = 'HuP51A'
adata61_out.obs['Sample'] = 'HuP61A'
adata74_out.obs['Sample'] = 'HuP74A'
adata93_out.obs['Sample'] = 'HuP93A'
adata97_out.obs['Sample'] = 'HuP97A'


adata51_out.obs['Health_Status'] = 'T2D'
adata61_out.obs['Health_Status'] = 'T2D'
adata74_out.obs['Health_Status'] = 'T2D'
adata93_out.obs['Health_Status'] = 'T2D'
adata97_out.obs['Health_Status'] = 'T2D'


adata51_out.obs['Sex'] = 'F'
adata61_out.obs['Sex'] = 'M'
adata74_out.obs['Sex'] = 'F'
adata93_out.obs['Sex'] = 'M'
adata97_out.obs['Sex'] = 'F'


adata51_out.obs['Age'] = '74'
adata61_out.obs['Age'] = '69'
adata74_out.obs['Age'] = '58'
adata93_out.obs['Age'] = '71'
adata97_out.obs['Age'] = '69'


adata51_out.obs['BMI'] = '15.1'
adata61_out.obs['BMI'] = '26.8'
adata74_out.obs['BMI'] = '32.9'
adata93_out.obs['BMI'] = '24.7'
adata97_out.obs['BMI'] = '31.2'


adata51_out.obs['Ethnicity'] = 'Caucasian'
adata61_out.obs['Ethnicity'] = 'Caucasian'
adata74_out.obs['Ethnicity'] = 'Caucasian'
adata93_out.obs['Ethnicity'] = 'Caucasian'
adata97_out.obs['Ethnicity'] = 'Caucasian'


In [ ]:
t2d_data = [adata61_in, adata97_in, adata51_in, adata93_in, adata74_in]

In [ ]:
t2d_islet_cells = sc.concat(t2d_data, index_unique='_')
t2d_islet_cells

In [ ]:
t2d_data_exo = [adata61_out, adata97_out, adata51_out, adata93_out, adata74_out]

In [ ]:
t2d_exo_cells = sc.concat(t2d_data_exo, index_unique='_')
t2d_exo_cells

In [ ]:
t2dalldata= [t2d_exo_cells, t2d_islet_cells]

In [ ]:
t2dalldata_cells = sc.concat(t2dalldata, index_unique='_')
t2dalldata_cells

In [ ]:
t2dexo_cells = sc.pp.subsample(t2d_exo_cells, fraction=0.028, copy=True)
t2dexo_cells

In [ ]:
alldata_2= [t2d_islet_cells, t2dexo_cells]

In [ ]:
adata = sc.concat(alldata_2, index_unique='_')
adata

In [ ]:
adata.write_h5ad('/Users/kmeneses/Desktop/updated_data/t2dcells_allsamples.h5ad')

In [ ]:
sc.pp.normalize_total(adata) # Normalize counts per cell
sc.pp.log1p(adata)

# SECTION 2 — CONCORD INTEGRATION & UMAP
 Runs Concord batch-corrected latent embedding across T2D
 donors using Sample as domain key.
 Computes UMAP from Concord latent space (Concord_UMAP).

 Leiden clustering (resolution=1.0) is applied on the
 Concord neighbor graph, followed by manual merging and
 renaming of clusters into broad cell types stored in:
   adata.obs["merged_cell_type"]
   adata.obs["specific_cell_type"]

 Cell types annotated:
  Beta, Alpha, Delta, PPY, Acinar, Ductal,
   Endothelial Cells, Mesenchymal, Immune Cells


In [ ]:
# Set device to cpu or to gpu (if your torch has been set up correctly to use GPU), for mac you can use either torch.device('mps') or torch.device('cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Initialize Concord with an AnnData object, skip input_feature to use all features, set preload_dense=False if your data is very large
# Provide 'domain_key' if integrating across batches, see below
cur_ccd = ccd.Concord(adata=adata, input_feature=None, domain_key='Sample', device=device, preload_dense=True) 

# Encode data, saving the latent embedding in adata.obsm['Concord']
cur_ccd.fit_transform(output_key='Concord')

In [ ]:
ccd.ul.run_umap(adata, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['Sample', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
adata.write_h5ad('/Users/kmeneses/Desktop/Figure2_Data/allT2DSamples_concord.h5ad')

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))

# -------------------------
# SAMPLE
# -------------------------
sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color="Sample",
    size=1,
    frameon=True,
    legend_loc="right margin",
    ax=axes[0],
    show=False,
    sort_order=True
)

# Rasterize points
for col in axes[0].collections:
    col.set_rasterized(True)

axes[0].set_title("Sample")

# -------------------------
# LOCATION
# -------------------------
sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color="Location",
    size=1,
    frameon=True,
    legend_loc="right margin",
    ax=axes[1],
    show=False,
    sort_order=True
)

# Rasterize points
for col in axes[1].collections:
    col.set_rasterized(True)

axes[1].set_title("Location")

# =========================
# CLEAN AXES
# =========================
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
plt.tight_layout()

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_T2Dallsamples"

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight",
    pad_inches=0,
    dpi=600
)

plt.savefig(
    f"{out_base}.svg",
    format="svg",
    bbox_inches="tight",
    pad_inches=0,
    dpi=600
)

plt.show()

In [ ]:
sc.pp.neighbors(adata, use_rep='Concord')
sc.tl.leiden(adata, resolution=1.0, key_added='concord_leiden')

In [ ]:
marker_genes_dict = {
    "B-cell": ["CD79A", "CD19"],
    "Macrophages": ["CCR3", "CXCR3", "C1QC"],
    "Cadherins": ["CDH1", "CDH5"],
    "Beta Cells": ["IAPP", "GCK", "PDX1", "MAFA", 'UCN3', 'GLP1R', 'MAFB', 'NEUROD1', 'NKX6-1', 'PAX4', 'SOX1', 'SDC4'],
    "Pericytes": ["CSPG4", "PDGFRB", "S100A1", "SYP", 'ACE2', 'F3', 'RGS5', 'SFN', 'TMEM123'],
    "Endothelial": ['PLVAP', 'PECAM1', 'VWF', 'HSPG2', 'FLT1', 'RAMP2', 'PCAT19', 'CD34', 'SOX18', 'EGFL7', 'IFI27', 'ENG', 'PLPP1', 
                     'CLDN5', 'CLEC14A', 'CD93', 'MRTFB', 'TGFBR2', 'ESAM', 'KDR', 'TCIM', 'LIFR', 'SPARCL1', 'ITGA6', 'ICAM1'],
    "Ductal Cells": ["CFTR", "KRT19", "DCDC2", 'PROM1', 'SOX9', 'SPP1', 'TUBB3', 'VAT1L'], 
    "Endocrine Markers": ['CHGA', 'UCHL1', 'ISL1', 'CCDC186', 'SIMC1', 'NOS1', 'ERLIN1'], 
    "Alpha Cells": ['GCG', 'BEND4', 'IRX1', 'IRX2', 'LOXL4', 'RPL14'], 
    "Delta Cells": ['SST', 'SSTR2', 'HHEX', 'BMI1', 'VIP', 'RSPH1'], 
    "Epsilon Cells": ['GHR', 'LGALS3BP', 'POGK', 'SERGEF', 'VTN'], 
    "PPY Cells": ['PPY', 'PAX6', 'SERTM1'], 
    "Nerve Cells": ['CBX2', 'CEACAM1', 'GRIA2', 'MNX1', 'NCAM1', 'NPY', 'TRPV1']
}

In [ ]:
sc.pl.embedding(
    adata,
    basis='Concord_UMAP',
    color=['concord_leiden', 'Location' ],
    frameon=False,
    ncols=3,
)

In [ ]:
adata.obs['concord_leiden'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata, "concord_leiden", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:
sc.pl.dotplot(adata, marker_genes_dict, groupby='concord_leiden', vmax=20)

In [ ]:
adata

In [ ]:


def merge_clusters(adata, cluster_column, merge_dict, new_column_name="merged_clusters"):
    # Create a new column with original cluster labels
    adata.obs[new_column_name] = adata.obs[cluster_column].astype(str)

    # Apply merging rules
    for old_clusters, new_label in merge_dict.items():
        adata.obs.loc[adata.obs[cluster_column].isin(old_clusters), new_column_name] = new_label

    return adata


In [ ]:


# Example usage:
# Define clusters to merge
merge_dict = {
    ('3', '5', '16', '9'): 'Alpha Cells', 
    ('13', '2', '14'): 'Beta Cells',
    ('0', '4'): 'Acinar Cells', 
    ('7', '8'): 'Endothelial Cells',
    ('11', '15'): 'Immune Cells',  
}

# Merge clusters
adata = merge_clusters(adata, cluster_column="concord_leiden", merge_dict=merge_dict, new_column_name="merged_cell_type")

# Visualize the updated clusters
print(adata.obs["merged_cell_type"].value_counts())

In [ ]:
sc.tl.rank_genes_groups(adata, "merged_cell_type", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:
import scanpy as sc

def rename_cell_categories(adata, column, rename_dict):
    """
    Rename categories in a specified column of an AnnData object.

    Parameters:
    - adata: AnnData object.
    - column: str, the column in adata.obs that contains categorical labels.
    - rename_dict: dict, mapping of old category names to new names.

    Returns:
    - Updated AnnData object with renamed categories.
    """
    if column in adata.obs:
        adata.obs[column] = adata.obs[column].astype("category")  # Ensure categorical dtype
        adata.obs[column] = adata.obs[column].cat.rename_categories(rename_dict)
    else:
        print(f"Column '{column}' not found in adata.obs.")
    
    return adata


In [ ]:

# Example usage:
rename_dict = {
    "1": "Ductal Cells",
    "6": "Mesenchymal", 
    "10": "Delta Cells",
    "12": "PPY Cells", 

}

adata = rename_cell_categories(adata, column="merged_cell_type", rename_dict=rename_dict)

# Check renamed categories
print(adata.obs["merged_cell_type"].cat.categories)

In [ ]:


# Example usage:
# Define clusters to merge
merge_dict = {
    ('3', '5', '16'): 'Alpha Cells', 
    ('13', '2'): 'Beta Cells',
    ('0', '4'): 'Acinar Cells', 
    ('7', '8'): 'Endothelial Cells',
    ('11', '15'): 'Immune Cells',  
}

# Merge clusters
adata = merge_clusters(adata, cluster_column="concord_leiden", merge_dict=merge_dict, new_column_name="specific_cell_type")

# Visualize the updated clusters
print(adata.obs["specific_cell_type"].value_counts())

In [ ]:
sc.tl.rank_genes_groups(adata, "specific_cell_type", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:

# Example usage:
rename_dict = {
    "1": "Ductal Cells",
    "6": "Mesenchymal Cells", 
    "10": "Delta Cells", 
    "9": "Edge Alpha Cells", 
    "14": "Special Beta Cells", 
    "12": "PPY Cells", 

}

adata = rename_cell_categories(adata, column="specific_cell_type", rename_dict=rename_dict)

# Check renamed categories
print(adata.obs["specific_cell_type"].cat.categories)

In [ ]:
ECM = ['SFN', 'COL4A3', 'COL18A1', 'COL15A1', 'COL5A3', 'COL6A3', 'COL6A2', 'COL5A2', 'COL1A1', 'COL1A2', 'COL6A1', 'COL3A1', 
       'LGALS4', 'LOXL4', 'SFRP2', 'SPARCL1', 'IGFBP7', 'AGPS', 'AGRN', 'CLASRP', 'LUM', 'MGP', 'AMBP', 'HSPG2', 
       'F13A1', 'ASPN', 'LAMC3', 'TINAGL1', 'VWA1', 'PCOLCE', 'MFGE8', 'FBLN2', 
       'LAMB2', 'FBN1', 'LAMA2', 'THBS1', 'VTN', 'FN1', 'FGG', 'FGB', 'FGA', 'LAMA5', 'COL16A1', 'COL11A1', 
       'COL4A2', 'COL4A1', 'COL2A1', 'OGN', 'EMILIN1', 'LAMA4', 'POSTN', 'FBN2', 'COL12A1', 'COL14A1', 'COL5A1', 'LAMC1', 'LAMC2',  'LAMB3', 'FBLN1',
        'LAMB1', 'LAMB4', 'LAMA3', 'SDC1']

In [ ]:
vasc_marker_genes_dict = {
    "Activated Stellate": ["PDGFRB", "COL1A1", "COL6A3", "DCN", "LAMA2", "LUM", "FN1", "MMP2"],
    "Quiescent Stellate": ["PDGFRB", "CSPG4", "COL4A1", "RGS5", "ACE2"],
    "Endothelial Cells": ["FLT1", "PLVAP", "VWF"],
    "Pericytes": ["CSPG4", "PDGFRB", "S100A1", "SYP", 'C1R', 'CALD1', 'RGS5'],
    "Immuno Associated Vasculature": ['ICAM1', 'C1R', 'AIF1', 'IL4', 'CXCR3', 'C1S'],
    "Endocrine Markers": ['CHGA', 'GCG', 'SST', 'PPY'], 
    "Nerve Cells": ['CBX2', 'CEACAM1', 'GRIA2', 'MNX1', 'NCAM1', 'NPY', 'TRPV1']
}

In [ ]:
ECM_dict = { 
    "Collagens": ['PCOLCE', 'COL1A1', 'COL14A1', 'COL5A3', 'COL1A2', 'COL6A2', 'COL3A1', 'COL18A1', 'COL4A2', 'COL11A1', 
       'COL5A1', 'COL16A1', 'COL5A2', 'COL15A1', 'COL4A1', 'COL4A3', 'COL12A1', 'COL6A3', 'COL2A1', 'COL6A1'],
    "Laminins": ['LAMA4', 'LAMA5', 'LAMC1', 'LAMC2', 'LAMB2', 'LAMB3', 'LAMB1', 'LAMA2', 'LAMC3', 'LAMB4', 'LAMA3'],
    "Others": ['FBLN1','FGB','FGA','FGG','FBLN2','VWA1','HSPG2','FBN2','SDC1','F13A1','AMBP','MFGE8','TINAGL1','SFRP2',
               'SPARCL1','EMILIN1','FN1','LUM','THBS1','LOXL4','CLASRP','OGN','VTN','MGP','AGPS','LGALS4','SFN','ASPN','AGRN','POSTN']
}

In [ ]:
adata.obs["specific_cell_type"]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

palette = {
    "Acinar Cells": "#e67bda",
    "Ductal Cells": "#c0c9c0",
    "Mesenchymal Cells": "#2416f0",
    "Endothelial Cells": "#33a02c",
    "Immune Cells": "#eff21c",
    "Beta Cells": "#e70f0f",
    "Alpha Cells": "#dd9014",
    "Delta Cells": "#8f3cdc",
    "PPY Cells": "#12ccec",

}
# -------------------------
# PLOT
# -------------------------
sc.pl.embedding(
    adata,
    basis='Concord_UMAP',
    color=['merged_cell_type' ],
    size=10,
    ncols=2,
    palette=palette,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)


# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_locationsamples1_vasc_T2D"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# -------------------------
# PLOT
# -------------------------
sc.pl.embedding(
    adata,
    basis='Concord_UMAP',
    color=['specific_cell_type', 'Location', 'Sample', 'merged_cell_type' ],
    size=10,
    ncols=2,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)


# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_locationsamples1_vasc_T2D"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
marker_genes_dict = {
    "Beta Cell": ["MAFA", "PDX1", "IAPP", "NKX6-1"],
    "Alpah Cell": ["GCG", "CD79A"],
    "Delta Cell": ["SST"],
    "Quiescent Stellate": ["CSPG4", "PDGFRB"],
    "Activated Stellate": ["COL6A1", "COL4A1"],
    "Endothelail Cells": ["PLVAP", "PECAM1"],
    "Immune Cells": ["C1QC", "CD19"],
    "Gamma Cells": ["PPY"],
    "Nerve Cells": ["NPY"],
}

In [ ]:
sc.pl.dotplot(adata,marker_genes_dict, groupby='merged_cell_type', vmax=10 )

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SMALL DOTPLOT
# =========================
dp = sc.pl.dotplot(
    adata,
    marker_genes_dict,
    groupby="merged_cell_type",

    vmax=2,
    use_raw=False,

    standard_scale="var",
    dendrogram=True,
    swap_axes=False,

    # MUCH SMALLER
    figsize=(6,3),

    cmap="magma",

    return_fig=True,
    show=False
)

# render
dp.make_figure()

# =========================
# ACCESS FIG
# =========================
fig = plt.gcf()

# =========================
# FORMAT
# =========================
for ax in fig.axes:

    ax.tick_params(
        axis="x",
        labelsize=5,
        rotation=90,
        length=2
    )

    ax.tick_params(
        axis="y",
        labelsize=5,
        length=2
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

# =========================
# TIGHTER LAYOUT
# =========================
fig.tight_layout(pad=0.2)

# =========================
# SAVE
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/"
    "vascdict_dotplotvascleiden_SMALL"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
sc.pl.dotplot(adata,marker_genes_dict, groupby='specific_cell_type', vmax=10 )

In [ ]:
adata.write_h5ad('/Users/kmeneses/Desktop/updated_data/t2dcells_allsamples_filteredlabeled.h5ad')

# SECTION 3 — ISLET ROI GEOMETRY QC
 Parses islet ROI polygon files (WKT format) from the T2D
 MERSCOPE dataset to compute per-islet spatial metrics:
   - islet_area_um2
   - islet_perimeter_um
   - circularity (4π·area / perimeter²)
   - centroid coordinates

 Results exported to CSV and plotted as a 4-panel QC figure.


In [ ]:
import pandas as pd
import numpy as np
import glob
from shapely import from_wkt

# -------------------------------------
# FOLDER WITH ALL ROI CSV FILES
# -------------------------------------

roi_folder = "/Volumes/Samsung_4TB/Islets2_ROI_CSV_Merscope/T2D/"

roi_files = glob.glob(f"{roi_folder}/*.csv")

print("Total ROI files:", len(roi_files))


# -------------------------------------
# STORAGE
# -------------------------------------

islet_results = []


# -------------------------------------
# LOOP THROUGH ROI FILES
# -------------------------------------

for f in roi_files:

    df = pd.read_csv(f)

    sample = df["dataset"].iloc[0]

    print("Processing:", sample)

    # detect ID column
    if "ROI" in df.columns:
        id_col = "ROI"
    elif "EntityID" in df.columns:
        id_col = "EntityID"
    else:
        raise ValueError(f"No ROI or EntityID column in {f}")

    # safe geometry parsing
    for i, row in df.iterrows():

        try:
            poly = from_wkt(row["geometry"])

            if poly is None or poly.is_empty:
                continue

            area = poly.area
            perimeter = poly.length
            centroid = poly.centroid

            islet_results.append({

                "Sample": sample,
                "ROI_ID": row[id_col],

                "islet_area_um2": area,
                "islet_perimeter_um": perimeter,

                "centroid_x": centroid.x,
                "centroid_y": centroid.y
            })

        except Exception as e:
            print(f"Skipping invalid ROI in {sample} row {i}")


# -------------------------------------
# COMBINE RESULTS
# -------------------------------------

islet_df = pd.DataFrame(islet_results)

print(islet_df.head())


# -------------------------------------
# SAVE OUTPUT
# -------------------------------------

out_path = "/Users/kmeneses/Desktop/islet_geometry_metricsT2D.csv"

islet_df.to_csv(out_path, index=False)

print("Saved:", out_path)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# =====================================
# STYLE
# =====================================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================
# LOAD
# =====================================
df = pd.read_csv(
    "/Users/kmeneses/Desktop/islet_geometry_metricsT2D.csv"
)

# =====================================
# CIRCULARITY
# =====================================
df["circularity"] = (
    4 * np.pi *
    df["islet_area_um2"] /
    (df["islet_perimeter_um"]**2)
)

# =====================================
# FIGURE
# =====================================
fig, axes = plt.subplots(
    2, 2,
    figsize=(7,6)
)

# =====================================
# AREA DISTRIBUTION
# =====================================
sns.histplot(
    df["islet_area_um2"],
    bins=50,
    ax=axes[0,0]
)

axes[0,0].set_xlabel("Islet area (µm²)")
axes[0,0].set_title("Area distribution")

# =====================================
# PERIMETER DISTRIBUTION
# =====================================
sns.histplot(
    df["islet_perimeter_um"],
    bins=50,
    ax=axes[0,1]
)

axes[0,1].set_xlabel("Islet perimeter (µm)")
axes[0,1].set_title("Perimeter distribution")

# =====================================
# AREA VS PERIMETER
# =====================================
sns.scatterplot(
    data=df,
    x="islet_area_um2",
    y="islet_perimeter_um",
    alpha=0.6,
    s=10,
    ax=axes[1,0]
)

axes[1,0].set_xlabel("Area (µm²)")
axes[1,0].set_ylabel("Perimeter (µm)")
axes[1,0].set_title("Geometry relationship")

# =====================================
# CIRCULARITY
# =====================================
sns.histplot(
    df["circularity"],
    bins=50,
    ax=axes[1,1]
)

axes[1,1].set_xlabel("Circularity")
axes[1,1].set_title("Shape QC")

# =====================================
# FORMAT
# =====================================
for ax in axes.flatten():

    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =====================================
# LAYOUT
# =====================================
plt.tight_layout()

# =====================================
# SAVE
# =====================================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/SI/"
    "T2D_islet_geometry_QC_panel"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ QC panel saved")

In [ ]:
adata.obs_names

In [ ]:
adata.obs_names = adata.obs_names.str.split("_").str[0]

In [ ]:
adata.obs_names

In [ ]:
coords = adata.obsm["spatial"]

In [ ]:
islet_df["circularity"] = (
    4 * np.pi * islet_df["islet_area_um2"] /
    (islet_df["islet_perimeter_um"] ** 2)
)

In [ ]:
adata.uns["islet_geometry"] = islet_df

In [ ]:
adata.uns["islet_geometry"]

# SECTION 4 — CELL POLYGON LOADING
 Loads cell boundary polygons from per-sample Parquet files
 (cell_boundaries.parquet) for each T2D donor.
 Handles multi-tile samples (HuP61A has 4 tiles).

 Geometry parsed from WKB or WKT depending on file format.
 Cell centroids computed from polygon centroids.
 Duplicate cells across tiles are deduplicated by EntityID.

 Key output:
   cell_polygons  — dict: EntityID → shapely polygon
   cell_centroids — dict: EntityID → (x, y) centroid


In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from shapely import from_wkb, from_wkt

# -----------------------------
# DEFINE PATHS FIRST (OUTSIDE LOOP)
# -----------------------------
sample_to_parquet = {
    "HuP61A": [
        "/Volumes/Samsung_4TB/HuP61A_data_/1/cell_boundaries.parquet",
        "/Volumes/Samsung_4TB/HuP61A_data_/2/cell_boundaries.parquet", 
        "/Volumes/Samsung_4TB/HuP61A_data_/3/cell_boundaries.parquet", 
        "/Volumes/Samsung_4TB/HuP61A_data_/4/cell_boundaries.parquet"
    ],
    "HuP20A": "/Volumes/Samsung_4TB/HuP97A_data/cell_boundaries.parquet",
    "HuP31A": "/Volumes/Samsung_4TB/HuP_93A_data/cell_boundaries.parquet",
    "HuP53A": "/Volumes/Samsung_4TB/HuP51A_data/cell_boundaries.parquet",
    "HuP71A": "/Volumes/Samsung_4TB/HuP74A_data/cell_boundaries.parquet",
}

# -----------------------------
# INIT STORAGE
# -----------------------------
cell_polygons = {}
cell_centroids = {}

print("Loading cell boundaries...")

# -----------------------------
# LOOP THROUGH FILES
# -----------------------------
for sample, paths in sample_to_parquet.items():

    if not isinstance(paths, list):
        paths = [paths]

    for path in paths:

        print("Reading:", path)

        table = pq.read_table(path).to_pandas()

        # find geometry column
        geom_candidates = [c for c in table.columns if "geom" in c.lower()]
        if len(geom_candidates) == 0:
            raise ValueError(f"No geometry column found in {path}")

        geom_col = geom_candidates[0]

        ids = table["EntityID"].astype(str)
        g = table[geom_col].dropna()

        if len(g) == 0:
            continue

        first_val = g.iloc[0]

        if isinstance(first_val, (bytes, bytearray, memoryview)):
            polys = from_wkb(g.values)
        else:
            polys = from_wkt(g.values)

        for eid, poly in zip(ids, polys):

            key = str(eid)

            # skip duplicates across tiles
            if key in cell_polygons:
                continue

            cell_polygons[key] = poly

            c = poly.centroid
            cell_centroids[key] = (c.x, c.y)

print("Loaded unique polygons:", len(cell_polygons))

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

from shapely import from_wkt
from shapely.strtree import STRtree

# =========================================================
# SETTINGS
# =========================================================
ROI_FOLDER = "/Volumes/Samsung_4TB/Islets2_ROI_CSV_Merscope/T2D"

# =========================================================
# FIND ROI FILES
# =========================================================
roi_files = glob.glob(
    f"{ROI_FOLDER}/*.csv"
)

print(f"Found {len(roi_files)} ROI files\n")

# =========================================================
# VALID SAMPLES
# =========================================================
valid_samples = set(
    adata.obs["Sample"].astype(str).unique()
)

print("Valid samples:")
print(valid_samples)

# =========================================================
# LOAD ROI POLYGONS
# =========================================================
sample_roi_lists = {}

for f in roi_files:

    fname = os.path.basename(f)

    print(f"\nProcessing: {fname}")

    # -------------------------------------------------
    # SAMPLE ID
    # -------------------------------------------------
    sample_id = (
        fname
        .replace(".csv", "")
        .split("_")[0]
    )

    print(f"Parsed sample: {sample_id}")

    if sample_id not in valid_samples:

        print("⚠ Not in adata")
        continue

    df = pd.read_csv(f)

    # -------------------------------------------------
    # FIND GEOMETRY COLUMN
    # -------------------------------------------------
    geom_cols = [
        c for c in df.columns
        if "geom" in c.lower()
    ]

    if len(geom_cols) == 0:

        print("⚠ No geometry column")
        continue

    geom_col = geom_cols[0]

    rois = []

    for val in df[geom_col]:

        if pd.isna(val):
            continue

        try:

            poly = from_wkt(val)

            if (
                poly.is_empty or
                (not poly.is_valid)
            ):
                continue

            rois.append(poly)

        except:
            continue

    print(f"Loaded {len(rois)} ROIs")

    if len(rois) > 0:

        sample_roi_lists[sample_id] = rois

# =========================================================
# BUILD TREES
# =========================================================
sample_trees = {
    s: STRtree(rois)
    for s, rois in sample_roi_lists.items()
}

print("\n✔ Trees built:", len(sample_trees))

# =========================================================
# COMPUTE TRUE EDGE DISTANCES
# =========================================================
distances = np.full(
    adata.n_obs,
    np.nan
)

for sample_id in adata.obs["Sample"].unique():

    if sample_id not in sample_trees:
        continue

    tree = sample_trees[sample_id]
    rois = sample_roi_lists[sample_id]

    idxs = np.where(
        adata.obs["Sample"] == sample_id
    )[0]

    print(
        f"Processing {len(idxs)} cells "
        f"for {sample_id}"
    )

    for idx in idxs:

        cid = str(adata.obs_names[idx])

        # -------------------------------------------------
        # GET CELL POLYGON
        # -------------------------------------------------
        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty or
            (not poly.is_valid)
        ):
            continue

        centroid = poly.centroid

        # -------------------------------------------------
        # FIND NEAREST ROI
        # -------------------------------------------------
        nearest = tree.nearest(centroid)

        # shapely 2 compatibility
        if hasattr(nearest, "geom_type"):

            roi = nearest

        else:

            roi = rois[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        dist = poly.distance(roi.boundary)

        # -------------------------------------------------
        # NEGATIVE IF INSIDE ROI
        # -------------------------------------------------
        if roi.contains(centroid):

            dist = -dist

        distances[idx] = dist

# =========================================================
# STORE
# =========================================================
adata.obs["dist_edge_islet_um"] = distances

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adata.obs["dist_edge_islet_um"]
    .describe()
)

inside_pct = (
    adata.obs["dist_edge_islet_um"] < 0
).mean() * 100

print(
    f"% inside islet: "
    f"{inside_pct:.2f}%"
)

In [ ]:
# fraction inside islets
inside_pct = (
    adata.obs["dist_edge_islet_um"] < 0
).mean() * 100

print("Inside islet %:", inside_pct)

# median inside distance
print(
    adata.obs.loc[
        adata.obs["dist_edge_islet_um"] < 0,
        "dist_edge_islet_um"
    ].median()
)

# median outside distance
print(
    adata.obs.loc[
        adata.obs["dist_edge_islet_um"] > 0,
        "dist_edge_islet_um"
    ].median()
)

In [ ]:
# =====================================================
# CHECK DISTANCE COVERAGE PER SAMPLE
# =====================================================

DIST_COL = "dist_edge_islet_um"

summary = (
    adata.obs
    .groupby("Sample")[DIST_COL]
    .agg(
        total_cells="size",
        non_nan=lambda x: x.notna().sum(),
        nan_count=lambda x: x.isna().sum()
    )
)

summary["percent_with_distance"] = (
    summary["non_nan"] /
    summary["total_cells"]
) * 100

print(summary)

# =====================================================
# SHOW ONLY PROBLEM DONORS
# =====================================================
print("\nDonors with missing distances:\n")

print(
    summary[
        summary["nan_count"] > 0
    ]
)

# SECTION 5 — SPATIAL DISTANCE CALCULATIONS
 Computes true edge-to-edge distances between each cell and
 anatomical landmarks using shapely STRtree spatial indexing.

 Distances computed per sample to avoid cross-donor leakage.
 Pixels converted to microns (scale factor: 0.108 µm/pixel).
 Metrics added to adata.obs:
   dist_edge_islet_um      — distance to nearest islet ROI boundary
                             (negative = inside islet)
   dist_edge_endo_um       — distance to nearest endothelial cell
   dist_edge_mesenchymal_um — distance to nearest mesenchymal cell
   dist_edge_beta_um        — distance to nearest beta cell

 Coverage statistics and per-donor NaN rates are checked
 after each computation.


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SMALLER FIGURE
# =========================
plt.rcParams["figure.figsize"] = (4, 2)

# =========================
# VIOLIN
# =========================
sc.pl.violin(
    adata,
    keys="dist_edge_islet_um",
    groupby="merged_cell_type",
    rotation=45,
    stripplot=False,   # 🔥 huge file-size reduction
    jitter=False,
    show=False
)

# =========================
# FORMAT
# =========================
fig = plt.gcf()

for ax in fig.axes:

    ax.set_ylabel("Distance to islet edge (µm)")

    ax.tick_params(
        axis="x",
        labelsize=6
    )

    ax.tick_params(
        axis="y",
        labelsize=6
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/SI/"
    "dist_to_islet_violin_by_mergedcelltype"
)

# PNG usually much smaller + cleaner
plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

# rasterized PDF smaller than SVG
plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

# optional SVG
plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
sc.pl.violin(
    adata,
    "dist_edge_islet_um",
    groupby="specific_cell_type",
    rotation=90
)

In [ ]:
sc.pl.violin(
    adata,
    "dist_edge_islet_um",
    groupby="merged_cell_type",
    rotation=90
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

d = adata.obs["dist_edge_islet_um"].dropna()

print("=== DISTANCE SUMMARY (µm) ===")
print(f"  Total cells:     {len(d)}")
print(f"  Inside islet  (<0): {(d < 0).sum()}  ({100*(d<0).mean():.1f}%)")
print(f"  On boundary  (=0):  {(d == 0).sum()}")
print(f"  Outside islet (>0): {(d > 0).sum()}  ({100*(d>0).mean():.1f}%)")
print(f"\n  Min:    {d.min():.2f} µm")
print(f"  Median: {d.median():.2f} µm")
print(f"  Max:    {d.max():.2f} µm")
print(f"  NaNs:   {adata.obs['dist_edge_islet_um'].isna().sum()}")

# Distribution plot
fig, figsize=(4, 2)

# Full distribution
axes[0].hist(d, bins=100, color="steelblue", edgecolor="none")
axes[0].axvline(0, color="red", linewidth=1, linestyle="--", label="islet boundary")
axes[0].set_xlabel("Distance to islet edge (µm)")
axes[0].set_ylabel("Cell count")
axes[0].set_title("Full distribution")
axes[0].legend()

# Zoomed: ±50 µm around boundary
zoomed = d[d.between(-50, 50)]
axes[1].hist(zoomed, bins=100, color="steelblue", edgecolor="none")
axes[1].axvline(0, color="red", linewidth=1, linestyle="--", label="islet boundary")
axes[1].set_xlabel("Distance to islet edge (µm)")
axes[1].set_title("Zoomed: ±50 µm")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_endo = np.full(
    adata.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in adata.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        adata.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        adata.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            adata.obs["merged_cell_type"]
            == "Endothelial Cells"
        )
    )

    peri_ids = (
        adata.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_endo[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
adata.obs[
    "dist_edge_endo_um"
] = dist_edge_endo

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adata.obs[
        "dist_edge_endo_um"
    ].describe()
)

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_endo = np.full(
    adata.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in adata.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        adata.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        adata.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            adata.obs["merged_cell_type"]
            == "Mesenchymal Cells"
        )
    )

    peri_ids = (
        adata.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_endo[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
adata.obs[
    "dist_edge_mesenchymal_um"
] = dist_edge_endo

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adata.obs[
        "dist_edge_mesenchymal_um"
    ].describe()
)

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_endo = np.full(
    adata.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in adata.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        adata.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        adata.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            adata.obs["merged_cell_type"]
            == "Beta Cells"
        )
    )

    peri_ids = (
        adata.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_endo[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
adata.obs[
    "dist_edge_beta_um"
] = dist_edge_endo

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    adata.obs[
        "dist_edge_beta_um"
    ].describe()
)

In [ ]:
group_col = "merged_cell_type"

vasc_t2d = adata[
    adata.obs[group_col].isin(["Mesenchymal Cells", "Endothelial Cells"])
].copy()

In [ ]:
vasc_t2d.write_h5ad('/Users/kmeneses/Desktop/updated_data/t2dvascells_allsamples_filteredlabeled.h5ad')

In [ ]:
vasc_t2d = sc.read_h5ad('/Users/kmeneses/Desktop/updated_data/t2dvascells_allsamples_filteredlabeled.h5ad')

# SECTION 6 — VASCULAR SUBTYPE REFINEMENT (T2D)
 Subsets mesenchymal and endothelial cells from adata into
 vasc_t2d for vascular-focused re-clustering.

 Leiden sub-clustering (resolution=1.0) on Concord neighbors,
 followed by manual merging based on marker gene expression:
   PECAM1, PLVAP, VWF      → Endothelial Cells
   RGS5, PDGFRB            → Pericytes
   COL1A1, DCN, LUM        → Fibroblasts

 Islet-located fibroblasts are relabeled as:
   "Islet-associated Fibroblasts"
 Final refined labels stored in:
   vasc_t2df.obs["cell_type"]

 UMAP recomputed on vascular subset (Concord_UMAPmRU).


In [ ]:
sc.pl.embedding(
    vasc_t2d,
    basis='Concord_UMAP',
    color=['merged_cell_type', 'Location'],
    frameon=False,
    ncols=3,
)

In [ ]:
sc.pp.neighbors(vasc_t2d, use_rep='Concord')
sc.tl.leiden(vasc_t2d, resolution=1.0, key_added='vasc_leiden')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# -------------------------
# PLOT
# -------------------------
sc.pl.embedding(
    vasc_t2d,
    basis='Concord_UMAP',
    color=['specific_cell_type', 'Location', 'Sample', 'vasc_leiden' ],
    size=10,
    ncols=2,
    frameon=False,
    legend_loc='right',
    show=False,
    sort_order=True
)


# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_locationsamples1_vasconly_T2D"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
sc.pl.embedding(
    vasc_t2d,
    basis='Concord_UMAP',
    color=['vasc_leiden', 'RGS5', 'PLVAP', 'DCN', 'COL1A1'],
    frameon=False,
    ncols=3,
    vmax=5
)

In [ ]:
vasc_t2d.obs['vasc_leiden'].value_counts()

In [ ]:
endo_peri_refined_markers = {
"ECs": ["PECAM1", "PLVAP", "VWF"],
"Pericytes": ["CSPG4", "PDGFRB", 'RGS5', 'DCN', "LUM"]
}

In [ ]:
vasc_t2df

In [ ]:
adata

In [ ]:
# ---------------------------------------
# MAKE COPY OF ORIGINAL LABELS
# ---------------------------------------

adata.obs["cell_type_refined"] = (
    adata.obs["merged_cell_type"].astype(str)
)

# ---------------------------------------
# TRANSFER VASCULAR LABELS
# ---------------------------------------
# assumes:
# - vasc_adata contains refined vascular labels
# - labels stored in:
#       vasc_adata.obs["cell_type_final"]
# - obs_names match between objects

adata.obs.loc[
    vasc_t2df.obs_names,
    "cell_type_refined"
] = (
    vasc_t2df.obs["cell_type"]
    .astype(str)
    .values
)

# ---------------------------------------
# OPTIONAL:
# REMOVE UNUSED CATEGORIES
# ---------------------------------------

adata.obs["cell_type_refined"] = (
    adata.obs["cell_type_refined"]
    .astype("category")
    .cat.remove_unused_categories()
)

# ---------------------------------------
# CHECK COUNTS
# ---------------------------------------

print(
    adata.obs["cell_type_refined"]
    .value_counts()
)

In [ ]:
import scanpy as sc

# ----------------------------------------
# RUN LEIDEN DIRECTLY IN MAIN ADATA
# ----------------------------------------

sc.pp.neighbors(
    adata,
    use_rep="Concord"
)

sc.tl.leiden(
    adata,
    restrict_to=("cell_type_refined", ["Mesenchymal Cells"]),
    resolution=0.5,
    key_added="remaining_mes_leiden"
)

# ----------------------------------------
# VISUALIZE
# ----------------------------------------

sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color=["remaining_mes_leiden"],
    legend_loc="on data"
)

# ----------------------------------------
# DGE ON SUBCLUSTERS
# ----------------------------------------

mes = adata[
    adata.obs["cell_type_refined"] == "Mesenchymal Cells"
].copy()

sc.tl.rank_genes_groups(
    mes,
    groupby="remaining_mes_leiden",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups_dotplot(
    mes,
    n_genes=8,
    standard_scale="var"
)

# ----------------------------------------
# KNOWN MARKERS
# ----------------------------------------

markers = [
    "PECAM1", "PLVAP", "VWF",
    "RGS5", "PDGFRB",
    "COL1A1", "DCN", "LUM",
    "PTPRC", "CD3D",
    "KRT19", "KRT18",
    "PRSS1", "CPA1"
]

markers = [g for g in markers if g in adata.var_names]

sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color=["remaining_mes_leiden"] + markers,
    ncols=4
)

In [ ]:
# remove tiny clusters before DGE

counts = mes.obs["remaining_mes_leiden"].value_counts()

keep_clusters = counts[counts > 1].index

mes = mes[
    mes.obs["remaining_mes_leiden"].isin(keep_clusters)
].copy()

# remove unused categories
mes.obs["remaining_mes_leiden"] = (
    mes.obs["remaining_mes_leiden"]
    .cat.remove_unused_categories()
)

# run DGE
sc.tl.rank_genes_groups(
    mes,
    groupby="remaining_mes_leiden",
    method="wilcoxon",
    use_raw=False
)

# visualize
sc.pl.rank_genes_groups_dotplot(
    mes,
    n_genes=8,
    standard_scale="var"
)

In [ ]:
sc.pl.dotplot(mes, endo_peri_refined_markers, groupby='remaining_mes_leiden')

In [ ]:
sc.pl.dotplot(adata, endo_peri_refined_markers, groupby='remaining_mes_leiden')

In [ ]:
# ----------------------------------------
# START FROM EXISTING REFINED LABELS
# ----------------------------------------

adata.obs["cell_type_merged"] = (
    adata.obs["cell_type_refined"].astype(str)
)

# ----------------------------------------
# REASSIGN REMAINING MESENCHYMAL CLUSTERS
# ----------------------------------------

merge_dict = {
    "Mesenchymal Cells,3": "Endothelial Cells",
    "Mesenchymal Cells,5": "Pericytes",
    "Mesenchymal Cells,0": "Fibroblasts",
    "Mesenchymal Cells,1": "Fibroblasts",
    "Mesenchymal Cells,2": "Fibroblasts",
    "Mesenchymal Cells,4": "Fibroblasts",
}

# only update cells that are in the dict
mask = adata.obs["remaining_mes_leiden"].astype(str).isin(merge_dict.keys())

adata.obs.loc[
    mask,
    "cell_type_merged"
] = (
    adata.obs.loc[mask, "remaining_mes_leiden"]
    .astype(str)
    .map(merge_dict)
    .values
)

# ----------------------------------------
# REMOVE UNUSED CATEGORIES
# ----------------------------------------

adata.obs["cell_type_merged"] = (
    adata.obs["cell_type_merged"]
    .astype("category")
    .cat.remove_unused_categories()
)

# ----------------------------------------
# CHECK COUNTS
# ----------------------------------------

print(
    adata.obs["cell_type_merged"]
    .value_counts()
)

In [ ]:
# remove remaining Mesenchymal Cells

adata = adata[
    adata.obs["cell_type_merged"] != "Mesenchymal Cells"
].copy()

# remove unused categories
adata.obs["cell_type_merged"] = (
    adata.obs["cell_type_merged"]
    .cat.remove_unused_categories()
)

# check counts
print(
    adata.obs["cell_type_merged"]
    .value_counts()
)

In [ ]:
sc.tl.rank_genes_groups(adata, "cell_type_merged", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:
sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color=["cell_type_merged", "Location"],
    ncols=4
)

In [ ]:
sc.pl.dotplot(vasc_t2d, endo_peri_refined_markers, groupby='vasc_leiden', vmax=2)

In [ ]:
sc.tl.rank_genes_groups(vasc_t2d, "vasc_leiden", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(vasc_t2d, n_genes=20)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import kruskal
import matplotlib as mpl

# -----------------------------------
# SETTINGS
# -----------------------------------
CLUSTER_COL = "vasc_leiden"

DIST_ISLET = "dist_edge_islet_um"
DIST_ENDO = "dist_edge_endo_um"

# -----------------------------------
# SUBSET
# -----------------------------------
df = vasc_t2df.obs[[
    CLUSTER_COL,
    DIST_ISLET,
    DIST_ENDO
]].copy()

# remove NA
df = df.dropna()

# ensure string labels
df[CLUSTER_COL] = df[CLUSTER_COL].astype(str)

# get cluster list automatically
clusters = sorted(df[CLUSTER_COL].unique())

# -----------------------------------
# SUMMARY STATS
# -----------------------------------
summary = (
    df.groupby(CLUSTER_COL)[[DIST_ISLET, DIST_ENDO]]
    .median()
    .sort_values(DIST_ISLET)
)

print("\nMedian distances:")
print(summary)

# -----------------------------------
# KRUSKAL TEST
# -----------------------------------
groups_islet = [
    df[df[CLUSTER_COL] == c][DIST_ISLET]
    for c in clusters
]

groups_endo = [
    df[df[CLUSTER_COL] == c][DIST_ENDO]
    for c in clusters
]

k1, p1 = kruskal(*groups_islet)
k2, p2 = kruskal(*groups_endo)

print("\nKruskal p-value (islet distance):", p1)
print("Kruskal p-value (endo distance):", p2)

# -----------------------------------
# PLOT: ISLET DISTANCE
# -----------------------------------
plt.figure(figsize=(10,4))

sns.boxplot(
    data=df,
    x=CLUSTER_COL,
    y=DIST_ISLET,
    showfliers=False
)

sns.stripplot(
    data=df,
    x=CLUSTER_COL,
    y=DIST_ISLET,
    color="black",
    size=1.5,
    alpha=0.2
)

plt.ylabel("Distance to islet boundary (µm)")
plt.xlabel("Cluster")
plt.title("Spatial distance to islet boundary")

plt.tight_layout()
plt.show()

# -----------------------------------
# PLOT: ENDOTHELIAL DISTANCE
# -----------------------------------
plt.figure(figsize=(10,4))

sns.boxplot(
    data=df,
    x=CLUSTER_COL,
    y=DIST_ENDO,
    showfliers=False
)

sns.stripplot(
    data=df,
    x=CLUSTER_COL,
    y=DIST_ENDO,
    color="black",
    size=1.5,
    alpha=0.2
)

plt.ylabel("Distance to endothelial cells (µm)")
plt.xlabel("Cluster")
plt.title("Spatial distance to endothelium")

plt.tight_layout()
plt.show()

In [ ]:
vasc_t2df.obs['vasc_leiden'].value_counts()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import matplotlib as mpl

# -----------------------------------
# STYLE
# -----------------------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# -----------------------------------
# SETTINGS
# -----------------------------------
CELLTYPE_COL = "cell_type"
DIST_ISLET = "dist_edge_islet_um"
DIST_ENDO = "dist_edge_endo_um"

groups = [
    "Fibroblasts",
    "Islet-associated Fibroblasts"
]

# -----------------------------------
# SUBSET
# -----------------------------------
df = vasc_t2df.obs[
    vasc_t2df.obs[CELLTYPE_COL].isin(groups)
][[
    CELLTYPE_COL,
    DIST_ISLET,
    DIST_ENDO
]].copy()

# remove NA
df = df.dropna()

# -----------------------------------
# SUMMARY STATS
# -----------------------------------
print("\nMedian distances:")
print(
    df.groupby(CELLTYPE_COL)[[DIST_ISLET, DIST_ENDO]]
    .median()
)

# -----------------------------------
# MWU TESTS
# -----------------------------------
fib = df[df[CELLTYPE_COL] == "Fibroblasts"]
isletfib = df[df[CELLTYPE_COL] == "Islet-associated Fibroblasts"]

# islet distance
u1, p1 = mannwhitneyu(
    fib[DIST_ISLET],
    isletfib[DIST_ISLET]
)

# endothelial distance
u2, p2 = mannwhitneyu(
    fib[DIST_ENDO],
    isletfib[DIST_ENDO]
)

print("\nDistance to islet boundary p-value:", p1)
print("Distance to endothelial cells p-value:", p2)

# -----------------------------------
# PLOT: ISLET DISTANCE
# -----------------------------------
plt.figure(figsize=(4,4))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_ISLET,
    showfliers=False
)

sns.stripplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_ISLET,
    color="black",
    size=2,
    alpha=0.3
)

plt.ylabel("Distance to islet boundary (µm)")
plt.xlabel("")
plt.title("Fibroblast spatial distribution")

plt.tight_layout()
plt.show()

# -----------------------------------
# PLOT: ENDOTHELIAL DISTANCE
# -----------------------------------
plt.figure(figsize=(4,4))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_ENDO,
    showfliers=False
)

sns.stripplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_ENDO,
    color="black",
    size=2,
    alpha=0.3
)

plt.ylabel("Distance to endothelial cells (µm)")
plt.xlabel("")
plt.title("Fibroblast vascular proximity")

plt.tight_layout()
plt.show()

In [ ]:

# Example usage:
# Define clusters to merge
merge_dict = {
    ('2', '5', '7', '10', '11'): 'Endothelial Cells',
    ('0', '6', '12'): 'Fibroblasts',
    ('1', '9'): 'Pericytes',
}

# Merge clusters
vasc_t2d = merge_clusters(vasc_t2d, cluster_column="vasc_leiden", merge_dict=merge_dict, new_column_name="cell_type")

# Visualize the updated clusters
print(vasc_t2d.obs["cell_type"].value_counts())

In [ ]:
sc.tl.rank_genes_groups(vasc_t2d, "cell_type", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(vasc_t2d, n_genes=20)

In [ ]:


# Example usage:
rename_dict = {
    "8": "Islet-associated Fibroblasts",
}

vasc_t2d = rename_cell_categories(vasc_t2d, column="cell_type", rename_dict=rename_dict)

# Check renamed categories
print(vasc_t2d.obs["cell_type"].cat.categories)

In [ ]:
vasc_t2df = vasc_t2d[
    ~vasc_t2d.obs["cell_type"].isin(["3", "4"])
].copy()

In [ ]:
sc.tl.rank_genes_groups(vasc_t2df, "cell_type", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(vasc_t2df, n_genes=20)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DEG
# =========================
sc.tl.rank_genes_groups(
    vasc_t2df,
    groupby="cell_type",
    use_raw=False,
    method="wilcoxon"
)

# =========================
# PLOT
# =========================
sc.pl.rank_genes_groups(
    vasc_t2df,
    n_genes=40,
    sharey=False,
    fontsize=7,
    show=False
)

# =========================
# ACCESS FIGURE
# =========================
fig = plt.gcf()

# =========================
# FORMAT
# =========================
for ax in fig.axes:

    ax.tick_params(
        axis="x",
        labelsize=8,
        rotation=90
    )

    ax.tick_params(
        axis="y",
        labelsize=8
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/"
    "rank_genes_groups_t2d"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
print(vasc_t2df.obs["cell_type"].value_counts())

In [ ]:
sc.pl.embedding(
    vasc_t2df,
    basis='Concord_UMAP',
    color=['cell_type' , 'Sample', 'Location'],
    frameon=False,
    ncols=2,
    size=10,
)

In [ ]:
ccd.ul.run_umap(vasc_t2df, source_key='Concord', result_key='Concord_UMAPmRU', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['cell_type', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    vasc_t2df, basis='Concord_UMAPmRU', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PALETTES
# =========================
cell_palette = {
    "Pericytes": "#FD15FA",
    "Endothelial Cells": "#08c93b",
    "Fibroblasts": "#0eecf8",
    "Islet-associated Fibroblasts": "#fe9e0f"    
}

location_palette = {
    "Islet": "#3B0A45",      # dark purple
    "Exocrine": "#CFA0E9"    # light purple
}

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(1, 2, figsize=(4, 2))

# ---- PANEL 1: CELL TYPE ----
sc.pl.embedding(
    vasc_t2df,
    basis="Concord_UMAPmRU",
    color="cell_type",
    palette=cell_palette,
    size=10,
    frameon=False,
    legend_loc="center left",
    ax=axes[0],
    show=False,
    sort_order=True
)

# ---- PANEL 2: LOCATION ----
sc.pl.embedding(
    vasc_t2df,
    basis="Concord_UMAPmRU",
    color="Location",
    palette=location_palette,
    size=5,
    frameon=False,
    wspace=0.5,
    legend_loc="right",
    ax=axes[1],
    show=False,
    sort_order=True
)

# =========================
# FORMAT
# =========================
for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points for small file size
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_vasc_celltype_locationT2D"

os.makedirs(os.path.dirname(out_base), exist_ok=True)

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")

plt.show()

print("✔ DONE — clean UMAP exported")

In [ ]:


# -------------------------
# PLOT
# -------------------------
sc.pl.embedding(
    vasc_t2df,
    basis="Concord_UMAPmRU",
    color=["Sample", "cell_type"],
    size=10,
    ncols=2,
    frameon=False,
    legend_loc='center left',
    show=False,
    sort_order=True
)


# =========================
# FORMAT ALL AXES
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_no93217162samples_filtered_locationsamples_vasc_T2D"

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
vasc_t2df.obs['cell_type'].value_counts()

# =============================================================
# SECTION 7 — VASCULAR DEG ANALYSIS (T2D)
# =============================================================
 Wilcoxon-based differential gene expression comparing
 vascular cell types within the T2D islet compartment.

 Comparisons run (use_raw=False):
   1. Pericytes vs Endothelial Cells   2. Islet-associated Fibroblasts vs Pericytes  3. Fibroblasts vs Islet-associated Fibroblasts (reversed)  4. Fibroblasts vs Islet-associated Fibroblasts

 Results exported as ranked gene group plots and CSVs.

 BM gene module expression visualized as:
    - dotplot (sc.pl.dotplot)
   - matrixplot (sc.pl.matrixplot)
 across Pericytes, Endothelial Cells, and
 Islet-associated Fibroblasts (islet only).

In [ ]:

fibro_islet = vasc_t2df.obs[
    (vasc_t2df.obs['cell_type'] == "Fibroblasts") &
    (vasc_t2df.obs['Location'] == "Islet")
]

# ------------------------------------------
# Print total count
# ------------------------------------------

print("Number of fibroblasts in islets:")
print(len(fibro_islet))

In [ ]:
# ------------------------------------------
# Subset fibroblasts only
# ------------------------------------------

fibro = vasc_t2df[
    vasc_t2df.obs["cell_type"] == "Fibroblasts"
].copy()

# make sure Location has both groups
print(fibro.obs["Location"].value_counts())

# ------------------------------------------
# Run DGE: Islet vs Exocrine fibroblasts
# ------------------------------------------

sc.tl.rank_genes_groups(
    fibro,
    groupby="Location",
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# ------------------------------------------
# Plot top DEGs
# ------------------------------------------

sc.pl.rank_genes_groups(
    fibro,
    n_genes=25,
    sharey=False,
    figsize=(10,6)
)

# ------------------------------------------
# Export DEG table
# ------------------------------------------

deg_fibro = sc.get.rank_genes_groups_df(
    fibro,
    group="Islet"
)

deg_fibro.head(20)

# optional save
deg_fibro.to_csv(
    "/Users/kmeneses/Desktop/islet_vs_exocrine_fibro_DEGs.csv",
    index=False
)

In [ ]:
# ------------------------------------------
# Subset the two fibroblast populations
# ------------------------------------------

fib_compare = vasc_t2df[
    vasc_t2df.obs["cell_type"].isin([
        "Fibroblasts",
        "Islet-associated Fibroblasts"
    ])
].copy()

# optional: only islet cells
fib_compare = fib_compare[
    fib_compare.obs["Location"] == "Islet"
].copy()

# ------------------------------------------
# DGE
# ------------------------------------------

sc.tl.rank_genes_groups(
    fib_compare,
    groupby="cell_type",
    groups=["Islet-associated Fibroblasts"],
    reference="Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

# ------------------------------------------
# Plot
# ------------------------------------------

sc.pl.rank_genes_groups(
    fib_compare,
    n_genes=25,
    figsize=(10,6),
    sharey=False
)

# ------------------------------------------
# Export DEGs
# ------------------------------------------

deg_islet_fib = sc.get.rank_genes_groups_df(
    fib_compare,
    group="Islet-associated Fibroblasts"
)

deg_islet_fib.head(25)

In [ ]:
# ------------------------------------------
# Subset the two fibroblast populations
# ------------------------------------------

fib_compare = vasc_t2df[
    vasc_t2df.obs["cell_type"].isin([
        "Fibroblasts",
        "Islet-associated Fibroblasts"
    ])
].copy()

# only islet cells
fib_compare = fib_compare[
    fib_compare.obs["Location"] == "Islet"
].copy()

# ------------------------------------------
# REVERSED DGE
# Fibroblasts vs Islet-associated Fibroblasts
# ------------------------------------------

sc.tl.rank_genes_groups(
    fib_compare,
    groupby="cell_type",
    groups=["Fibroblasts"],
    reference="Islet-associated Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

# ------------------------------------------
# Plot
# ------------------------------------------

sc.pl.rank_genes_groups(
    fib_compare,
    n_genes=25,
    figsize=(10,6),
    sharey=False
)

# ------------------------------------------
# Export DEGs
# ------------------------------------------

deg_fib = sc.get.rank_genes_groups_df(
    fib_compare,
    group="Fibroblasts"
)

deg_fib.head(25)

In [ ]:
# ==========================================
# MERGE ONLY ISLET FIBROBLASTS
# INTO "Islet-associated Fibroblasts"
# ==========================================

CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"

# ------------------------------------------
# Convert to string temporarily
# ------------------------------------------

vasc_t2df.obs[CELLTYPE_COL] = (
    vasc_t2df.obs[CELLTYPE_COL]
    .astype(str)
)

# ------------------------------------------
# Rename ONLY fibroblasts located in islets
# ------------------------------------------

vasc_t2df.obs.loc[
    (vasc_t2df.obs[CELLTYPE_COL] == "Fibroblasts") &
    (vasc_t2df.obs[LOCATION_COL] == "Islet"),
    CELLTYPE_COL
] = "Islet-associated Fibroblasts"

# ------------------------------------------
# Convert back to category
# ------------------------------------------

vasc_t2df.obs[CELLTYPE_COL] = (
    vasc_t2df.obs[CELLTYPE_COL]
    .astype("category")
)

# ------------------------------------------
# Check counts
# ------------------------------------------

print(
    vasc_t2df.obs[CELLTYPE_COL]
    .value_counts()
)

In [ ]:
sc.tl.rank_genes_groups(
    vasc_t2df,
    groupby="cell_type",
    groups=["Pericytes"],      # ✅ exact match
    reference="Endothelial Cells",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups(vasc_t2df, n_genes=20)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# OUTPUT PATH
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/DEG_pericytes_vs_fibroT2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    vasc_t2df,
    groupby="cell_type",
    groups=["Endothelial Cells"],      # ✅ exact match
    reference="Pericytes",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups(vasc_t2df, n_genes=20, show=False)
# =========================
# GET FIG + CLEAN
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.tick_params(labelsize=7)
    
    # rasterize heavy elements (keeps SVG small)
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE HIGH RES
# =========================
fig.savefig(out_base + "pervsecs.png", dpi=600, bbox_inches="tight")
fig.savefig(out_base + "pervsecs.pdf", bbox_inches="tight")
fig.savefig(out_base + "pervsecs.svg", dpi=600, bbox_inches="tight")

plt.show()

print("✔ DONE — high-res DEG plot saved")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# OUTPUT PATH
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/DEG_pericytes_vs_PIFfibroT2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    vasc_t2df,
    groupby="cell_type",
    groups=["Islet-associated Fibroblasts"],      # ✅ exact match
    reference="Pericytes",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups(vasc_t2df, n_genes=20, show=False)
# =========================
# GET FIG + CLEAN
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.tick_params(labelsize=7)
    
    # rasterize heavy elements (keeps SVG small)
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE HIGH RES
# =========================
fig.savefig(out_base + "pervsecs.png", dpi=600, bbox_inches="tight")
fig.savefig(out_base + "pervsecs.pdf", bbox_inches="tight")
fig.savefig(out_base + "pervsecs.svg", dpi=600, bbox_inches="tight")

plt.show()

print("✔ DONE — high-res DEG plot saved")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# OUTPUT PATH
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/DEG_firbo_vs_PIFfibroT2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    vasc_t2df,
    groupby="cell_type",
    groups=["Islet-associated Fibroblasts"],      # ✅ exact match
    reference="Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups(vasc_t2df, n_genes=20, show=False)
# =========================
# GET FIG + CLEAN
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.tick_params(labelsize=7)
    
    # rasterize heavy elements (keeps SVG small)
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE HIGH RES
# =========================
fig.savefig(out_base + "pervsecs.png", dpi=600, bbox_inches="tight")
fig.savefig(out_base + "pervsecs.pdf", bbox_inches="tight")
fig.savefig(out_base + "pervsecs.svg", dpi=600, bbox_inches="tight")

plt.show()

print("✔ DONE — high-res DEG plot saved")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# OUTPUT PATH
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/DEG_firbo_vs_PIFfibro_rvT2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    vasc_t2df,
    groupby="cell_type",
    groups=["Fibroblasts"],      # ✅ exact match
    reference="Islet-associated Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups(vasc_t2df, n_genes=20, show=False)
# =========================
# GET FIG + CLEAN
# =========================
fig = plt.gcf()
axes = fig.axes

for ax in axes:
    ax.set_title("")
    ax.tick_params(labelsize=7)
    
    # rasterize heavy elements (keeps SVG small)
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE HIGH RES
# =========================
fig.savefig(out_base + "pervsecs.png", dpi=600, bbox_inches="tight")
fig.savefig(out_base + "pervsecs.pdf", bbox_inches="tight")
fig.savefig(out_base + "pervsecs.svg", dpi=600, bbox_inches="tight")

plt.show()

print("✔ DONE — high-res DEG plot saved")

In [ ]:


# =========================
# STYLE
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "font.weight": "bold",
    "axes.titlesize": 9,
    "font.style": "normal",
    "font.variant": "normal",
    "font.stretch": "normal",  
    "axes.labelsize": 8,
    "axes.titleweight": "bold",
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SUBSET
# =========================
CELLTYPE_COL = "cell_type"
HEALTH_COL = "Health_Status"

vascular_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts", 
    "Fibroblasts"
]

adata_plot = vasc_t2df[
    (vasc_t2df.obs[CELLTYPE_COL].isin(vascular_types))
].copy()

adata_plot.obs[CELLTYPE_COL] = (
    adata_plot.obs[CELLTYPE_COL].cat.remove_unused_categories()
)

# =========================
# VALID GENES (DICT)
# =========================
valid_gene_dict = {}

for group, genes in endo_peri_refined_markers.items():
    valid = [g for g in genes if g in adata_plot.var_names]
    if len(valid) > 0:
        valid_gene_dict[group] = valid
    else:
        print(f"⚠️ {group} has no valid genes")

print(valid_gene_dict)

# =========================
# DOTPLOT (NO plt.subplots!)
# =========================
sc.pl.dotplot(
    adata_plot,
    var_names=valid_gene_dict,
    groupby=CELLTYPE_COL,
    vmax=4,
    cmap="magma",
    swap_axes=True,
    show=False
)

# 🔥 grab the correct figure
fig = plt.gcf()

# set size AFTER plotting
fig.set_size_inches(3, 3)

# optional: clean title
fig.suptitle("")

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/dotplotvasc_t2d"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

fig.savefig(f"{out_base}.pdf")      # best for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600)      # optional



In [ ]:
gene_mask = vasc_t2df.var_names.isin(ECM)
vasc_t2df.var['gene_list_mask'] = gene_mask

In [ ]:
bm_genes = [
    "COL4A1","COL4A2","COL4A3",
    "LAMA4","LAMA5",
    "LAMB1","LAMB2",
    "LAMC1","LAMC3"
]

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# ---------------------------------------------------
# STYLE
# ---------------------------------------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ---------------------------------------------------
# COLUMNS
# ---------------------------------------------------
CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"

groups = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts"
]

# ---------------------------------------------------
# GENES (ensure present)
# ---------------------------------------------------
bm_genes_use = [g for g in bm_genes if g in vasc_t2df.var_names]

print("Genes used:", len(bm_genes_use))

# ---------------------------------------------------
# SUBSET (ISLET ONLY)
# ---------------------------------------------------
ad = vasc_t2df[
    (vasc_t2df.obs[LOCATION_COL] == "Islet") &
    (vasc_t2df.obs[CELLTYPE_COL].isin(groups))
].copy()

# clean labels
ad.obs[CELLTYPE_COL] = ad.obs[CELLTYPE_COL].astype(str).str.strip()

print("Cells kept:", ad.n_obs)
print(ad.obs[CELLTYPE_COL].value_counts())

# ---------------------------------------------------
# EXTRACT EXPRESSION (🔥 THIS WAS MISSING)
# ---------------------------------------------------
X = sc.get.obs_df(
    ad,
    keys=bm_genes_use,
    use_raw=False,
    layer='counts' # change to "log1p" if you have it
)

X[CELLTYPE_COL] = ad.obs[CELLTYPE_COL].values

# ---------------------------------------------------
# MEAN EXPRESSION
# ---------------------------------------------------
mean_exp = X.groupby(CELLTYPE_COL).mean()

# safe ordering
order = [g for g in groups if g in mean_exp.index]
mean_exp = mean_exp.loc[order]

# ---------------------------------------------------
# 🔥 TRANSPOSE → genes on Y axis
# ---------------------------------------------------
plot_df = mean_exp

# ---------------------------------------------------
# PLOT
# ---------------------------------------------------
plt.figure(figsize=(4,2))

ax = sns.heatmap(
    plot_df,
    cmap="viridis",
    linewidths=0.5,
    cbar_kws={"label": "Mean expression"},
    vmin=0,
    vmax=2
)

# -------------------------
# AXIS FORMATTING
# -------------------------
ax.set_xlabel("Cell type")
ax.set_ylabel("Genes")

ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

ax.tick_params(axis='x', labelsize=7)
ax.tick_params(axis='y', labelsize=7)

plt.title("BM gene expression\nIslet vascular compartment (T2D)")

plt.tight_layout()

# ---------------------------------------------------
# SAVE
# ---------------------------------------------------
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/BM_pericyte_vs_peri_fibro_heatmap_islet_t2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")

plt.show()

print("✔ DONE — heatmap saved")

In [ ]:
sc.pl.dotplot(ad, bm_genes, groupby='cell_type', vmax=1, use_raw=False, cmap='magma')

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# ---------------------------------------------------
# Illustrator settings
# ---------------------------------------------------

mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"
HEALTH_COL = "Health_Status"

groups = [
    "Pericytes",
    "Islet-associated Fibroblasts",
    "Endothelial Cells"
]

# ---------------------------------------------------
# Colors for cell types
# ---------------------------------------------------



# ---------------------------------------------------
# BM genes
# ---------------------------------------------------

genes_plot = bm_genes

# ---------------------------------------------------
# Subset dataset
# ---------------------------------------------------

ad = vasc_t2df[
    (vasc_t2df.obs[LOCATION_COL] == "Islet") &
    (vasc_t2df.obs[CELLTYPE_COL].isin(groups))
].copy()

# ---------------------------------------------------
# Set category order
# ---------------------------------------------------




plt.figure(figsize=(2,3))


dp = sc.pl.dotplot(
    ad,
    genes_plot,
    groupby=CELLTYPE_COL,
    use_raw=False,
    cmap="magma",
    return_fig=True, 
    vmax=1,
    swap_axes=True
)

# ---------------------------------------------------
# Save
# ---------------------------------------------------

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SIBM_dotplot_pericyte_vs_periislefibo"

os.makedirs(os.path.dirname(out_base), exist_ok=True)

dp.savefig(f"{out_base}.svg", dpi=600)
dp.savefig(f"{out_base}.png", dpi=600)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# GLOBAL STYLE (LOCKED)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA (🔥 SUBSET HERE)
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "merged_cell_type"


df = adata.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

# =========================
# FIXED FIGURE SIZE (CRITICAL)
# =========================
fig, ax = plt.subplots(figsize=(3, 2))

# =========================
# PLOT
# =========================
sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    bins=100,
    element="step",
    stat="density",
    common_norm=False,
    linewidth=1,
    ax=ax
)

# vertical boundary line
ax.axvline(0, color="black", linestyle="--", linewidth=1)

# axes formatting
ax.set_xlabel("Distance to islet boundary (µm)")
ax.set_ylabel("Density")
ax.set_title("Spatial distribution relative to islet boundary")

# cleaner look
sns.despine(ax=ax)

# legend formatting
leg = ax.get_legend()
if leg is not None:
    leg.set_title("")
    for text in leg.get_texts():
        text.set_fontsize(7)

plt.tight_layout()

# =========================
# EXPORT
# =========================
plt.savefig("/Users/kmeneses/Desktop/Figure4_plots/SI/dist_islet_hist_allcellsT2D.svg",
            format="svg", dpi=600, bbox_inches="tight")

# SECTION 8 — SPATIAL ENRICHMENT ANALYSIS (T2D ISLET)
 Quantifies spatial proximity of T2D islet vascular cells to anatomical landmarks using precomputed distance columns.

# Analyses:
   1. KDE and boxplots: dist_edge_islet_um per cell type
   2. Pericyte vs Islet-associated Fibroblast distance
      to islet boundary — MWU + donor-level paired plots
   3. Pericyte–EC proximity: fraction within ±5 µm
   4. Pericyte coverage of endothelium (dist_edge_pericyte_um)
      — donor-level coverage fraction
   5. BM gene scores (COL4A, LAMA, LAMB, LAMC)
      compared between pericytes vs fibroblasts and
      pericytes vs endothelial cells (MWU + paired Wilcoxon)


In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# GLOBAL STYLE (LOCKED)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA (🔥 SUBSET HERE)
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "merged_cell_type"



df = adata.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

# =========================
# FIXED FIGURE SIZE (CRITICAL)
# =========================
fig, ax = plt.subplots(figsize=(3, 2))

# =========================
# PLOT
# =========================
sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    bins=100,
    element="step",
    stat="density",
    common_norm=False,
    linewidth=1,
    ax=ax
)

# vertical boundary line
ax.set_xlim(-20, 20)

ax.axvline(0, color="black", linestyle="--", linewidth=1)

# axes formatting
ax.set_xlabel("Distance to islet boundary (µm)")
ax.set_ylabel("Density")
ax.set_title("Spatial distribution relative to islet boundary")

# cleaner look
sns.despine(ax=ax)

# legend formatting
leg = ax.get_legend()
if leg is not None:
    leg.set_title("")
    for text in leg.get_texts():
        text.set_fontsize(7)

plt.tight_layout()

# =========================
# EXPORT
# =========================
plt.savefig("/Users/kmeneses/Desktop/Figure4_plots/SI/dist_islet_hist_allcells20_20_T2D.svg",
            format="svg", dpi=600, bbox_inches="tight")

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# GLOBAL STYLE (LOCKED)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "font.weight": "bold",
    "axes.titlesize": 9,
    "font.style": "normal",
    "font.variant": "normal",
    "font.stretch": "normal",  
    "axes.labelsize": 8,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA (🔥 CORRECT SUBSET)
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

cell_types = ["Fibroblasts", "Islet-associated Fibroblasts"]


df = vasc_t2df.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

# 🔥 ACTUAL FILTER (this was missing)
df = df[df[CELLTYPE_COL].isin(cell_types)]

# clean categories
df[CELLTYPE_COL] = df[CELLTYPE_COL].astype("category")
df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.remove_unused_categories()

# optional: control plotting order (nice for layering)
df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.reorder_categories(
    ["Fibroblasts", "Islet-associated Fibroblasts"],
    ordered=True
)

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(figsize=(3, 2))

# =========================
# COLORS
# =========================
palette = {
    "Fibroblasts": "#8C0EDA",
    "Islet-associated Fibroblasts": "#f9a942", 
}

# =========================
# PLOT
# =========================
sns.kdeplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    common_norm=False,
    linewidth=1,
    fill=True,
    alpha=0.4,
    ax=ax
)

# boundary line

ax.axvline(0, color="black", linestyle="--", linewidth=1)

# axes
ax.set_xlabel("Distance to Islet Boundary (µm)")
ax.set_ylabel("Cell Density")
ax.set_title("Spatial enrichment at islet boundary")

# clean style
sns.despine(ax=ax)

# legend
leg = ax.get_legend()
if leg is not None:
    leg.set_title("")
    for text in leg.get_texts():
        text.set_fontsize(7)

plt.tight_layout()

# =========================
# EXPORT (🔥 reduce SVG weight)
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/dist_islet_kde_vasc_2T2D.svg",
    format="svg",
    bbox_inches="tight",
    dpi=600   # 🔥 lower than 600 → lighter file
)

plt.show()

In [ ]:
adata.obs['merged_cell_type']

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# GLOBAL STYLE (LOCKED)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "font.weight": "bold",
    "axes.titlesize": 9,
    "font.style": "normal",
    "font.variant": "normal",
    "font.stretch": "normal",  
    "axes.labelsize": 8,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA (🔥 CORRECT SUBSET)
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

cell_types = ["Endothelial Cells", "Islet-associated Fibroblasts", "Pericytes"]


df = vasc_t2df.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

# 🔥 ACTUAL FILTER (this was missing)
df = df[df[CELLTYPE_COL].isin(cell_types)]

# clean categories
df[CELLTYPE_COL] = df[CELLTYPE_COL].astype("category")
df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.remove_unused_categories()



# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(figsize=(3, 2))

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942", 
    "Endothelial Cells": "#08c93b"
}

# =========================
# PLOT
# =========================
sns.kdeplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    common_norm=False,
    linewidth=1,
    fill=True,
    alpha=0.4,
    ax=ax
)

# boundary line

ax.axvline(0, color="black", linestyle="--", linewidth=1)

# axes
ax.set_xlabel("Distance to Islet Boundary (µm)")
ax.set_ylabel("Cell Density")
ax.set_title("Spatial enrichment at islet boundary")

# clean style
sns.despine(ax=ax)

# legend
leg = ax.get_legend()
if leg is not None:
    leg.set_title("")
    for text in leg.get_texts():
        text.set_fontsize(7)

plt.tight_layout()

# =========================
# EXPORT (🔥 reduce SVG weight)
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/dist_islet_kde_vascularcells_T2D.svg",
    format="svg",
    bbox_inches="tight",
    dpi=600   # 🔥 lower than 600 → lighter file
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts"
]

# =========================
# DATA
# =========================
df = islet_adata_vasc.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()

# spatial window (important)

df = df[df[CELLTYPE_COL].isin(cell_types)]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
}

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,2))

sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    bins=40,
    stat="density",        # 🔥 IMPORTANT (not counts)
    common_norm=False,     # 🔥 each cell type normalized independently
    palette=palette,
    alpha=0.6,
     # cleaner than filled bars
    linewidth=1
)

# boundary line
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Relative cell density")
plt.title("Spatial distribution near islet boundary")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out = "/Users/kmeneses/Desktop/Figure4_plots/SI/hist_boundary_distribution"

plt.savefig(out + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out + ".svg", bbox_inches="tight")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# GLOBAL STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "font.weight": "bold",
    "axes.titlesize": 9,
    "font.style": "normal",
    "font.variant": "normal",
    "font.stretch": "normal",
    "axes.labelsize": 8,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"

cell_types = [
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Pericytes"
]

# =====================================================
# KEEP NEEDED COLUMNS
# =====================================================
df = vasc_t2df.obs[
    [DIST_COL, CELLTYPE_COL, LOCATION_COL]
].dropna().copy()

# =====================================================
# SUBSET TO ISLET CELLS
# =====================================================
df = df[
    df[LOCATION_COL] == "Islet"
]

# =====================================================
# SUBSET CELL TYPES
# =====================================================
df = df[
    df[CELLTYPE_COL].isin(cell_types)
]

# clean categories
df[CELLTYPE_COL] = (
    df[CELLTYPE_COL]
    .astype("category")
)

df[CELLTYPE_COL] = (
    df[CELLTYPE_COL]
    .cat.remove_unused_categories()
)

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(
    figsize=(3, 2)
)

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
}

# =========================
# KDE PLOT
# =========================
sns.kdeplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    common_norm=False,
    linewidth=1,
    fill=True,
    alpha=0.4,
    ax=ax
)

# boundary line
ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

# axes
ax.set_xlabel(
    "Distance to Islet Boundary (µm)"
)

ax.set_ylabel(
    "Cell Density"
)

ax.set_title(
    "Spatial enrichment at islet boundary"
)

# clean style
sns.despine(ax=ax)

# legend
leg = ax.get_legend()

if leg is not None:

    leg.set_title("")

    for text in leg.get_texts():

        text.set_fontsize(7)

plt.tight_layout()

# =========================
# EXPORT
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/dist_islet_kde_vascularcells_T2D_islet.svg",
    format="svg",
    bbox_inches="tight",
    dpi=600
)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# GLOBAL STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "font.weight": "bold",
    "axes.titlesize": 9,
    "font.style": "normal",
    "font.variant": "normal",
    "font.stretch": "normal",
    "axes.labelsize": 8,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "lines.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"

cell_types = [
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Pericytes"
]

# =====================================================
# KEEP NEEDED COLUMNS
# =====================================================
df = islet_adata_vasc.obs[
    [DIST_COL, CELLTYPE_COL, LOCATION_COL]
].dropna().copy()

# =====================================================
# SUBSET TO ISLET CELLS
# =====================================================
df = df[
    df[LOCATION_COL] == "Islet"
]

# =====================================================
# SUBSET CELL TYPES
# =====================================================
df = df[
    df[CELLTYPE_COL].isin(cell_types)
]

# clean categories
df[CELLTYPE_COL] = (
    df[CELLTYPE_COL]
    .astype("category")
)

df[CELLTYPE_COL] = (
    df[CELLTYPE_COL]
    .cat.remove_unused_categories()
)

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(
    figsize=(3, 2)
)

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
}

# =========================
# KDE PLOT
# =========================
sns.kdeplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    common_norm=False,
    linewidth=1,
    fill=True,
    alpha=0.4,
    ax=ax
)

# boundary line
ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

# axes
ax.set_xlabel(
    "Distance to Islet Boundary (µm)"
)

ax.set_ylabel(
    "Cell Density"
)

ax.set_title(
    "Spatial enrichment at islet boundary"
)

# clean style
sns.despine(ax=ax)

# legend
leg = ax.get_legend()

if leg is not None:

    leg.set_title("")

    for text in leg.get_texts():

        text.set_fontsize(7)

plt.tight_layout()

# =========================
# EXPORT
# =========================
plt.savefig(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/dist_islet_kde_vascularcells_T2D_islet_isl.svg",
    format="svg",
    bbox_inches="tight",
    dpi=600
)

plt.show()

In [ ]:
# % within ±5 µm of boundary
df["near_boundary"] = df["dist_edge_islet_um"].abs() < 5

summary = (
    df.groupby("cell_type")["near_boundary"]
    .mean() * 100
)
print(summary)

In [ ]:
import scanpy as sc

CELLTYPE_COL = "Location"

vascular_types = [
    "Islet"
]

# ---------------------------------------------------
# subset vaascular cells
# ---------------------------------------------------
islet_adata_vasc = vasc_t2df[vasc_t2df.obs[CELLTYPE_COL].isin(vascular_types)].copy()

In [ ]:
islet_adata_vasc.obs['cell_type']

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/pericyte_vs_fib_distanceT2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

cell_types = [
    "Pericytes",
    "Islet-associated Fibroblasts"
]

# =========================
# SUBSET
# =========================
df = islet_adata_vasc.obs[
    (islet_adata_vasc.obs[CELLTYPE_COL].isin(cell_types))
][[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.remove_unused_categories()

# =========================
# PLOT
# =========================
plt.figure(figsize=(2,2))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    showfliers=False,
    boxprops=dict(facecolor="white", edgecolor="black"),
    medianprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black")
)

sns.stripplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    color="black",
    size=2,
    alpha=0.3,
    jitter=0.15
)

plt.ylabel("Distance to islet boundary (µm)")
plt.xlabel("")
# (optional) remove title for figure panels
# plt.title("Pericytes vs peri-islet fibroblasts spatial position")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/pericyte_vs_fibVsECS_distanceT2D"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

cell_types = [
    "Pericytes",
    "Islet-associated Fibroblasts",
    "Endothelial Cells"
]

# =========================
# SUBSET
# =========================
df = islet_adata_vasc.obs[
    (islet_adata_vasc.obs[CELLTYPE_COL].isin(cell_types))
][[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.remove_unused_categories()

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    showfliers=False,
    boxprops=dict(facecolor="white", edgecolor="black"),
    medianprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black")
)

sns.stripplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    color="black",
    size=2,
    alpha=0.3,
    jitter=0.15
)

plt.ylabel("Distance to islet boundary (µm)")
plt.xlabel("")
# (optional) remove title for figure panels
# plt.title("Pericytes vs peri-islet fibroblasts spatial position")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/pericyte_vs_fibVsECS_distanceT2_vasc_t2df"
os.makedirs(os.path.dirname(out_base), exist_ok=True)

cell_types = [
    "Pericytes",
    "Islet-associated Fibroblasts",
    "Endothelial Cells"
]

# =========================
# SUBSET
# =========================
df = vasc_t2df.obs[
    (vasc_t2df.obs[CELLTYPE_COL].isin(cell_types))
][[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.remove_unused_categories()

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    showfliers=False,
    boxprops=dict(facecolor="white", edgecolor="black"),
    medianprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black")
)

sns.stripplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    color="black",
    size=2,
    alpha=0.3,
    jitter=0.15
)

plt.ylabel("Distance to islet boundary (µm)")
plt.xlabel("")
# (optional) remove title for figure panels
# plt.title("Pericytes vs peri-islet fibroblasts spatial position")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", format="svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type"

cell_types = [
    "Pericytes",
    "Islet-associated Fibroblasts", 
    "Endothelial Cells", 
    "Beta Cells"
]

# subset data
df = islet_adata_vasc.obs[
    (islet_adata_vasc.obs[CELLTYPE_COL].isin(cell_types))
][[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].cat.remove_unused_categories()

plt.figure(figsize=(6,4))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    showfliers=False
)

sns.stripplot(
    data=df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    color="black",
    size=2,
    alpha=0.3
)


plt.ylabel("Distance to islet boundary (µm)")
plt.xlabel("")
plt.title("Pericytes vs peri-islet fibroblasts spatial position")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------
# column names
# -----------------------------
CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"
HEALTH_COL = "Health_Status"
DIST_COL = "dist_edge_endo_um"


# -----------------------------
# format data for Prism
# -----------------------------
peri = islet_adata_vasc.obs.loc[
    islet_adata_vasc.obs[CELLTYPE_COL] == "Pericytes",
    DIST_COL
].reset_index(drop=True)

peri_islet_fib = islet_adata_vasc.obs.loc[
    islet_adata_vasc.obs[CELLTYPE_COL] == "Islet-associated Fibroblasts",
    DIST_COL
].reset_index(drop=True)

prism_df = pd.DataFrame({
    "Pericytes": peri,
    "Islet-associated Fibroblasts": peri_islet_fib
})

# -----------------------------
# save for Prism
# -----------------------------
prism_df.to_csv(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/pericyte_vs_peri_islet_fib_distance_to_endo_prismT2D.csv",
    index=False
)

# -----------------------------
# plot
# -----------------------------
plt.figure(figsize=(4,4))

sns.violinplot(
    data=islet_adata_vasc.obs,
    x=CELLTYPE_COL,
    y=DIST_COL,
    order=["Pericytes","Islet-associated Fibroblasts"],
    inner="box",
    cut=0
)

plt.ylabel("Distance to Endothelial Cells (µm)")
plt.xlabel("")
plt.title("Distance to Endothelium (T2D Islet)")
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# -----------------------------
# column names
# -----------------------------
CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"
HEALTH_COL = "Health_Status"
DIST_COL = "dist_edge_endo_um"
SAMPLE_COL = "Sample"

cell_types = ["Pericytes", "Islet-associated Fibroblasts"]

# -----------------------------
# subset: Healthy Islet only
# -----------------------------


# -----------------------------
# donor-level aggregation
# -----------------------------
donor_df = (
    islet_adata_vasc.obs
    .groupby([SAMPLE_COL, CELLTYPE_COL])[DIST_COL]
    .mean()
    .reset_index()
)

# -----------------------------
# reshape for paired stats
# -----------------------------
paired = donor_df.pivot(
    index=SAMPLE_COL,
    columns=CELLTYPE_COL,
    values=DIST_COL
).dropna()

print(paired)

# -----------------------------
# paired test (correct!)
# -----------------------------
stat, pval = wilcoxon(
    paired["Pericytes"],
    paired["Islet-associated Fibroblasts"]
)

print(f"Wilcoxon p-value: {pval:.4f}")

# -----------------------------
# Prism export (paired format)
# -----------------------------
paired.to_csv(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/pericyte_vs_peri_islet_fib_donor_levelT2D.csv"
)

# -----------------------------
# plot (donor-level)
# -----------------------------
plt.figure(figsize=(4,4))

sns.boxplot(
    data=donor_df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    order=cell_types,
    showfliers=False
)

sns.stripplot(
    data=donor_df,
    x=CELLTYPE_COL,
    y=DIST_COL,
    order=cell_types,
    color="black",
    size=5
)

# OPTIONAL: connect paired donors (very nice)
for _, row in paired.iterrows():
    plt.plot(
        ["Pericytes", "Islet-associated Fibroblasts"],
        [row["Pericytes"], row["Islet-associated Fibroblasts"]],
        color="gray",
        alpha=0.5
    )

plt.ylabel("Distance to Endothelium (µm)")
plt.xlabel("")
plt.title("Donor-level comparison (T2D Islet)")

plt.tight_layout()
plt.show()

In [ ]:
adata

In [ ]:
import numpy as np
from shapely.strtree import STRtree

# =========================================================
# INIT
# =========================================================
dist_edge_pericyte = np.full(
    islet_adata_vasc.n_obs,
    np.nan
)

# =========================================================
# LOOP THROUGH SAMPLES
# =========================================================
for sample in islet_adata_vasc.obs["Sample"].unique():

    print("Processing:", sample)

    # -------------------------------------------------
    # SAMPLE MASK
    # -------------------------------------------------
    mask = (
        islet_adata_vasc.obs["Sample"] == sample
    )

    idx_in_adata = np.where(mask)[0]

    cells_in_sample = (
        islet_adata_vasc.obs_names[mask]
        .astype(str)
    )

    # -------------------------------------------------
    # PERICYTES IN SAMPLE
    # -------------------------------------------------
    peri_mask = (
        mask &
        (
            islet_adata_vasc.obs["cell_type"]
            == "Pericytes"
        )
    )

    peri_ids = (
        islet_adata_vasc.obs_names[peri_mask]
        .astype(str)
    )

    peri_polys = [
        cell_polygons[c]
        for c in peri_ids
        if c in cell_polygons
    ]

    if len(peri_polys) == 0:

        print(
            f"  No pericytes in {sample}. "
            f"Skipping."
        )

        continue

    # -------------------------------------------------
    # BUILD TREE
    # -------------------------------------------------
    tree = STRtree(peri_polys)

    peri_geoms_arr = np.array(
        peri_polys,
        dtype=object
    )

    # -------------------------------------------------
    # COMPUTE DISTANCES
    # -------------------------------------------------
    for i, cid in zip(
        idx_in_adata,
        cells_in_sample
    ):

        poly = cell_polygons.get(cid)

        if (
            poly is None or
            poly.is_empty
        ):
            continue

        nearest = tree.nearest(poly)

        if nearest is None:
            continue

        # shapely compatibility
        if hasattr(nearest, "geom_type"):

            nearest_geom = nearest

        else:

            nearest_geom = peri_geoms_arr[nearest]

        # -------------------------------------------------
        # TRUE EDGE-TO-EDGE DISTANCE
        # -------------------------------------------------
        d = poly.distance(nearest_geom)

        # convert pixels -> µm
        dist_edge_pericyte[i] = d * 0.108

# =========================================================
# STORE
# =========================================================
islet_adata_vasc.obs[
    "dist_edge_pericyte_um"
] = dist_edge_pericyte

print("\n✔ DONE")

# =========================================================
# SANITY CHECK
# =========================================================
print(
    islet_adata_vasc.obs[
        "dist_edge_pericyte_um"
    ].describe()
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE (Illustrator-friendly)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PARAMETERS
# =========================
CELLTYPE_COL = "cell_type"
HEALTH_COL = "Health_Status"
LOCATION_COL = "Location"
DIST_COL = "dist_edge_pericyte_um"
SAMPLE_COL = "Sample"

THRESH = 5  # µm threshold

# =========================
# SUBSET: Healthy Islet only
# =========================
df = islet_adata_vasc.obs.copy()



# =========================
# ENDOTHELIAL CELLS ONLY
# =========================
endo = df[df[CELLTYPE_COL] == "Endothelial Cells"].copy()
endo = endo.dropna(subset=[DIST_COL])

# =========================
# COMPUTE COVERAGE
# =========================
endo["covered"] = endo[DIST_COL] <= THRESH

coverage_fraction = endo["covered"].mean()
print(f"Overall pericyte coverage (≤ {THRESH} µm): {coverage_fraction:.3f}")

# =========================
# DONOR-LEVEL COVERAGE
# =========================
donor_cov = (
    endo.groupby(SAMPLE_COL)["covered"]
    .mean()
    .reset_index(name="coverage")
)

print("\nDonor-level coverage:")
print(donor_cov)

# =========================
# PLOT
# =========================
plt.figure(figsize=(2.5, 3))

sns.boxplot(
    data=donor_cov,
    y="coverage",
    width=0.5,
    showcaps=True,
    boxprops=dict(facecolor="white", edgecolor="black"),
    medianprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black")
)

sns.stripplot(
    data=donor_cov,
    y="coverage",
    color="black",
    size=4,
    jitter=0.1
)

plt.ylabel(f"Pericyte coverage fraction\n(≤ {THRESH} µm)")
plt.xlabel("")
plt.title("Islet vascular coverage (Healthy)")

sns.despine()
plt.tight_layout()

# =========================
# EXPORT
# =========================
out = "/Users/kmeneses/Desktop/Figure4_plots/SI/pericyte_coverage_T2D"

plt.savefig(f"{out}.pdf", dpi=600, bbox_inches="tight")
plt.savefig(f"{out}.svg", format="svg", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# PARAMETERS
# =========================
CELLTYPE_COL = "cell_type"
HEALTH_COL = "Health_Status"
LOCATION_COL = "Location"
DIST_COL = "dist_edge_endo_um"   # 🔥 distance from each cell → nearest endothelial cell
SAMPLE_COL = "Sample"

THRESH = 2  # µm

# =========================
# SUBSET: Healthy Islet only
# =========================


# =========================
# PERICYTES ONLY
# =========================
peri = df[df[CELLTYPE_COL] == "Pericytes"].copy()

# drop missing values
peri = peri.dropna(subset=[DIST_COL])

# =========================
# COMPUTE COVERAGE
# =========================
peri["covered"] = peri[DIST_COL] <= THRESH

coverage_fraction = peri["covered"].mean()
print(f"Pericytes within {THRESH} µm of EC: {coverage_fraction:.3f}")

# =========================
# DONOR-LEVEL COVERAGE
# =========================
donor_cov = (
    peri.groupby(SAMPLE_COL)["covered"]
    .mean()
    .reset_index(name="coverage")
)

print("\nDonor-level coverage:")
print(donor_cov)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=donor_cov,
    y="coverage",
    color="lightgray",
    width=0.5
)

sns.stripplot(
    data=donor_cov,
    y="coverage",
    color="black",
    size=5
)

plt.ylabel("Pericytes within 2 µm of EC (fraction)")
plt.xlabel("")
plt.title("Pericyte–endothelial proximity (Healthy Islet)")

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
islet_adata_vasc.obs['cell_type'].value_counts(
    
)

In [ ]:
sns.violinplot(
    data=donor_cov,
    y="coverage",
    color="lightgray",
    inner=None
)

sns.stripplot(
    data=donor_cov,
    y="coverage",
    color="black",
    size=5
)

In [ ]:
islet_adata_vasc

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# MATRIXPLOT
# =========================
sc.pl.matrixplot(
    islet_adata_vasc,
    bm_genes,
    groupby="cell_type",
    vmax=1,
    use_raw=False,
    cmap="magma",
    figsize=(3,2),
    swap_axes=True,
    dendrogram=False,
    show=False
)

# IMPORTANT:
# matrixplot creates its own figure,
# so use plt.gcf() AFTER plotting
fig = plt.gcf()

# =========================
# FORMAT
# =========================
for ax in fig.axes:

    ax.tick_params(
        axis="x",
        labelsize=7,
        rotation=90
    )

    ax.tick_params(
        axis="y",
        labelsize=7
    )

# =========================
# LAYOUT
# =========================
plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/SI/"
    "BM_matrixplot_celltypes"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# MATRIXPLOT
# =========================
mp = sc.pl.matrixplot(
    islet_adata_vasc,
    bm_genes,
    groupby="cell_type",
    use_raw=False,
    vmax=1,
    cmap="magma",

    # gene block organization
    var_group_rotation=0,

    figsize=(5,1),

    return_fig=True,
    show=False
)

# IMPORTANT
mp.make_figure()

# =========================
# ACCESS FIGURE
# =========================
fig = plt.gcf()

# =========================
# FORMAT
# =========================
for ax in fig.axes:

    ax.tick_params(
        axis="x",
        labelsize=6,
        rotation=90
    )

    ax.tick_params(
        axis="y",
        labelsize=6
    )

    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# LAYOUT
# =========================
fig.tight_layout()

# =========================
# SAVE
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/"
    "Pericyte_ECM_matrixplot"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
sc.pl.matrixplot(islet_adata_vasc, bm_genes, groupby='cell_type', vmax=1, use_raw=False, cmap='magma')